# Amazon ML Challenge 2026 — Business Entity Resolution
## Raw audit and information-preserving preprocessing

This notebook uses only competition-provided files. It does not build a matcher, use external data, geocode, or alter original TSV files. Run cells 1–12 first, review the generated audit/collision reports, then explicitly enable conversion in Cell 2 and continue from Cell 16. The first pass stops before TSV-to-Parquet conversion.

In [24]:
from pathlib import Path
import collections, csv, hashlib, json, math, os, re, sqlite3, statistics, unicodedata
import numpy as np
import pandas as pd

DATA_ROOT = Path('/content/dataset')
OUTPUT_ROOT = Path('/content/entity_resolution_processed')
CHUNK_SIZE = 100_000
RANDOM_SEED = 2026
VALIDATION_FRACTION = 0.15
RUN_CONVERSION = True # After reviewing the completed audit, set True and rerun Cell 16 onward.
SAMPLE_EVERY = 50       # Deterministic sample interval for descriptive quantiles/collision audits.
REQUIRED_SOURCE_COLUMNS = ['entity_id', 'business_name', 'business_address', 'country']
SOURCE_FILES = {
    'train_source1': DATA_ROOT / 'train/train_source1.tsv',
    'train_source2': DATA_ROOT / 'train/train_source2.tsv',
    'train_source3': DATA_ROOT / 'train/train_source3.tsv',
    'test_source1': DATA_ROOT / 'test/test_source1.tsv',
    'test_source2': DATA_ROOT / 'test/test_source2.tsv',
    'test_source3': DATA_ROOT / 'test/test_source3.tsv',
}
GROUND_TRUTH_FILE = DATA_ROOT / 'train/train_ground_truth.tsv'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Data:', DATA_ROOT, '| Output:', OUTPUT_ROOT, '| Conversion enabled:', RUN_CONVERSION)

Data: /content/dataset | Output: /content/entity_resolution_processed | Conversion enabled: True


## 1. Verify the dataset structure and required files
Fail if a required input is absent. `.DS_Store` and other unrelated files are ignored.

In [10]:
expected_dirs = [DATA_ROOT/'train', DATA_ROOT/'test']
for d in expected_dirs:
    if not d.is_dir(): raise FileNotFoundError(f'Missing directory: {d}')
for label, path in SOURCE_FILES.items():
    if not path.is_file(): raise FileNotFoundError(f'Missing {label}: {path}')
if not GROUND_TRUTH_FILE.is_file(): raise FileNotFoundError(f'Missing ground truth: {GROUND_TRUTH_FILE}')
print('All seven required TSV files exist.')
for label,path in {**SOURCE_FILES, 'train_ground_truth':GROUND_TRUTH_FILE}.items():
    print(f'{label:20s} {path.stat().st_size/1024**3:7.3f} GiB')

All seven required TSV files exist.
train_source1          0.196 GiB
train_source2          0.456 GiB
train_source3          0.469 GiB
test_source1           0.163 GiB
test_source2           0.474 GiB
test_source3           0.471 GiB
train_ground_truth     0.118 GiB


In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Strict TSV readers and schema checks
Use `sep="\t"`, UTF-8, string dtype, and `keep_default_na=False`. Empty cells remain empty strings; literal `NA`, `null`, and `-` are not automatically converted to missing. Malformed TSV rows raise errors.

In [11]:
GT_COLUMNS = ['source1_entity_id', 'matched_entity_ids']
def header(path):
    return pd.read_csv(path, sep='\t', dtype='string', encoding='utf-8',
        keep_default_na=False, na_filter=False, nrows=0, on_bad_lines='error').columns.tolist()
def chunks(path, usecols=None):
    return pd.read_csv(path, sep='\t', dtype='string', encoding='utf-8',
        keep_default_na=False, na_filter=False, chunksize=CHUNK_SIZE,
        usecols=usecols, on_bad_lines='error')
SCHEMAS = {}
for label,path in {**SOURCE_FILES, 'train_ground_truth':GROUND_TRUTH_FILE}.items():
    cols=header(path); need=GT_COLUMNS if label=='train_ground_truth' else REQUIRED_SOURCE_COLUMNS
    missing=sorted(set(need)-set(cols))
    if missing: raise ValueError(f'{label}: missing required columns {missing}; found {cols}')
    SCHEMAS[label]=cols
    print(label, cols)

train_source1 ['entity_id', 'business_name', 'business_address', 'country']
train_source2 ['entity_id', 'business_name', 'business_address', 'country']
train_source3 ['entity_id', 'business_name', 'business_address', 'country']
test_source1 ['entity_id', 'business_name', 'business_address', 'country']
test_source2 ['entity_id', 'business_name', 'business_address', 'country']
test_source3 ['entity_id', 'business_name', 'business_address', 'country']
train_ground_truth ['source1_entity_id', 'matched_entity_ids']


## 3. Raw source audit
Exact row, empty-field, country, prefix, and ID counts are accumulated by chunk. Length/token percentiles and example lists use deterministic bounded samples. ID uniqueness is checked with an on-disk SQLite index to avoid retaining all IDs in RAM. Nothing is dropped or fixed.

In [12]:
AUDIT_DB=OUTPUT_ROOT/'audit_ids.sqlite'
if AUDIT_DB.exists(): AUDIT_DB.unlink()
db=sqlite3.connect(AUDIT_DB)
db.execute('CREATE TABLE ids (entity_id TEXT PRIMARY KEY, source TEXT NOT NULL)')
db.execute('CREATE INDEX ids_source ON ids(source)')
def source_code(label): return 'S'+label[-1]
def audit_source(label,path):
    d={'rows':0,'columns':SCHEMAS[label], 'missing':collections.Counter(),
       'literal_placeholders':collections.Counter(), 'prefixes':collections.Counter(),
       'malformed_id_rows':0,'empty_id_rows':0,'duplicate_id_rows':0,
       'leading_trailing_whitespace':collections.Counter(), 'countries':collections.Counter(),
       'name_lengths':[],'address_lengths':[],'name_tokens':[],'address_tokens':[],
       'numeric_tokens':collections.Counter(),'non_ascii':collections.Counter(),
       'non_latin_sample_count':collections.Counter(),'short_names':[], 'short_addresses':[],
       'long_names':[], 'long_addresses':[], 'unicode_examples':[], 'placeholder_examples':collections.defaultdict(list)}
    src=source_code(label); expected=src+'-'; running=0
    placeholders=['NA','N/A','NULL','null','-','unknown','UNKNOWN']
    for df in chunks(path):
        n=len(df); d['rows']+=n
        for col in REQUIRED_SOURCE_COLUMNS:
            s=df[col].astype(str); blank=s.str.strip().eq('')
            d['missing'][col]+=int(blank.sum())
            d['leading_trailing_whitespace'][col]+=int(s.ne(s.str.strip()).sum())
            for token in placeholders:
                mask=s.str.strip().eq(token); d['literal_placeholders'][col+'='+token]+=int(mask.sum())
                if mask.any() and len(d['placeholder_examples'][col+'='+token])<5:
                    d['placeholder_examples'][col+'='+token].extend(s[mask].head(5-len(d['placeholder_examples'][col+'='+token])).tolist())
        ids=df.entity_id.astype(str); d['empty_id_rows']+=int(ids.str.strip().eq('').sum())
        prefixes=ids.str.extract(r'^([A-Za-z0-9]+-)')[0].fillna('<none>')
        d['prefixes'].update(prefixes.value_counts().to_dict())
        valid=ids.str.fullmatch(re.escape(expected)+r'[^\s,;]+',na=False)
        d['malformed_id_rows']+=int((~valid & ids.str.strip().ne('')).sum())
        batch=[(x,src) for x in ids if x.strip()]
        before=db.total_changes; db.executemany('INSERT OR IGNORE INTO ids VALUES (?,?)',batch)
        d['duplicate_id_rows']+=len(batch)-(db.total_changes-before); db.commit()
        d['countries'].update(df.country.astype(str).str.strip().value_counts().to_dict())
        for col,key in [('business_name','name'),('business_address','address')]:
            vals=df[col].astype(str); nonblank=vals.str.strip().ne(''); sample=vals[nonblank].iloc[::SAMPLE_EVERY]
            d[key+'_lengths'].extend(sample.str.len().tolist())
            d[key+'_tokens'].extend(sample.str.findall(r'\w+').str.len().tolist())
            d['numeric_tokens'][key]+=int(vals.str.findall(r'(?<!\w)\d+(?!\w)').str.len().sum())
            non=vals.str.contains(r'[^\x00-\x7f]',regex=True); d['non_ascii'][key]+=int(non.sum())
            for v in vals[non].head(8):
                item={'field':key,'value':v[:200]}
                if len(d['unicode_examples'])<16 and item not in d['unicode_examples']: d['unicode_examples'].append(item)
            for v in sample:
                if any(unicodedata.category(ch).startswith('L') and ord(ch)>127 and 'LATIN' not in unicodedata.name(ch,'') for ch in v):
                    d['non_latin_sample_count'][key]+=1
            short=vals[nonblank & vals.str.len().le(2)].head(10).tolist()
            long=vals[vals.str.len().ge(300)].head(10).tolist()
            target=d['short_names'] if key=='name' else d['short_addresses']; target.extend(x for x in short if x not in target)
            target=d['long_names'] if key=='name' else d['long_addresses']; target.extend(x[:200] for x in long if x[:200] not in target)
        running+=n
        if running % (CHUNK_SIZE*10)==0: print(label, f'{running:,} rows audited')
    for key in ['name_lengths','address_lengths','name_tokens','address_tokens']:
        a=np.asarray(d[key],dtype=float)
        d[key+'_sample_quantiles']={str(q):float(np.quantile(a,q)) for q in [0,.5,.9,.99,1]} if len(a) else {}
        del d[key]
    d['unique_ids_in_source']=db.execute('SELECT count(*) FROM ids WHERE source=?',(src,)).fetchone()[0]
    return d
SOURCE_AUDIT={}
for label,path in SOURCE_FILES.items():
    print('Auditing',label)
    SOURCE_AUDIT[label]=audit_source(label,path)
    print(json.dumps(SOURCE_AUDIT[label],ensure_ascii=False,indent=2))

Auditing train_source1
train_source1 1,000,000 rows audited
train_source1 2,000,000 rows audited
{
  "rows": 2206821,
  "columns": [
    "entity_id",
    "business_name",
    "business_address",
    "country"
  ],
  "missing": {
    "entity_id": 0,
    "business_name": 0,
    "business_address": 0,
    "country": 0
  },
  "literal_placeholders": {
    "entity_id=NA": 0,
    "entity_id=N/A": 0,
    "entity_id=NULL": 0,
    "entity_id=null": 0,
    "entity_id=-": 0,
    "entity_id=unknown": 0,
    "entity_id=UNKNOWN": 0,
    "business_name=NA": 0,
    "business_name=N/A": 0,
    "business_name=NULL": 0,
    "business_name=null": 0,
    "business_name=-": 0,
    "business_name=unknown": 0,
    "business_name=UNKNOWN": 0,
    "business_address=NA": 0,
    "business_address=N/A": 0,
    "business_address=NULL": 0,
    "business_address=null": 0,
    "business_address=-": 0,
    "business_address=unknown": 0,
    "business_address=UNKNOWN": 0,
    "country=NA": 0,
    "country=N/A": 0,
    "

## 4. Ground-truth cardinality and reference audit
An empty `matched_entity_ids` is an explicit zero-match label. A SQLite table stores every reference and checks that each target exists in the expected S2/S3 source. No label is removed.

In [13]:
db.execute('CREATE TABLE gt_rows (source1_id TEXT, raw_matches TEXT)')
db.execute('CREATE TABLE gt_refs (source1_id TEXT, target_id TEXT, expected_source TEXT)')
gt=collections.Counter(); bad_examples=[]
for df in chunks(GROUND_TRUTH_FILE):
    pairs=[]; refs=[]
    for sid,raw in zip(df.source1_entity_id.astype(str),df.matched_entity_ids.astype(str)):
        gt['rows']+=1; matches=[x.strip() for x in raw.split(',') if x.strip()]
        gt['zero_matches' if len(matches)==0 else 'one_match' if len(matches)==1 else 'multiple_matches']+=1
        pairs.append((sid,raw))
        for rid in matches: refs.append((sid,rid,'S2' if rid.startswith('S2-') else 'S3' if rid.startswith('S3-') else ''))
    db.executemany('INSERT INTO gt_rows VALUES (?,?)',pairs); db.executemany('INSERT INTO gt_refs VALUES (?,?,?)',refs); db.commit()
gt['duplicate_source1_label_rows']=db.execute('SELECT coalesce(sum(n-1),0) FROM (SELECT count(*) n FROM gt_rows GROUP BY source1_id HAVING n>1)').fetchone()[0]
gt['missing_source1_ids']=db.execute('SELECT count(*) FROM (SELECT DISTINCT source1_id FROM gt_rows) g LEFT JOIN ids i ON i.entity_id=g.source1_id AND i.source="S1" WHERE i.entity_id IS NULL').fetchone()[0]
gt['bad_target_references']=db.execute('SELECT count(*) FROM gt_refs r LEFT JOIN ids i ON i.entity_id=r.target_id AND i.source=r.expected_source WHERE r.expected_source="" OR i.entity_id IS NULL').fetchone()[0]
bad_examples=db.execute('SELECT r.source1_id,r.target_id,r.expected_source,i.source FROM gt_refs r LEFT JOIN ids i ON i.entity_id=r.target_id WHERE r.expected_source="" OR i.entity_id IS NULL LIMIT 25').fetchall()
print('Ground-truth audit:',dict(gt)); print('Bad reference examples:',bad_examples)
print('Train S1 rows:',SOURCE_AUDIT['train_source1']['rows'],'| ground-truth rows:',gt['rows'])

Ground-truth audit: {'rows': 2206821, 'multiple_matches': 1964417, 'zero_matches': 123247, 'one_match': 119157, 'duplicate_source1_label_rows': 0, 'missing_source1_ids': 0, 'bad_target_references': 0}
Bad reference examples: []
Train S1 rows: 2206821 | ground-truth rows: 2206821


## 5. Open-set country and schema audit
Countries are discovered from file contents. No country list is hard-coded into transformations. France in test is retained. This cell prints observed country values and counts per file.

In [14]:
for label,d in SOURCE_AUDIT.items():
    print(label, 'country values/counts:', json.dumps(d['countries'],ensure_ascii=False))
    unexpected=[p for p in d['prefixes'] if p not in {'S1-','S2-','S3-'}]
    if unexpected: print('Unexpected prefixes:',unexpected)
print('Schema differences:', {k:v for k,v in SCHEMAS.items()})

train_source1 country values/counts: {"US": 1323633, "India": 883188}
train_source2 country values/counts: {"US": 3016817, "India": 2017799}
train_source3 country values/counts: {"US": 3170056, "India": 2115547}
test_source1 country values/counts: {"India": 809986, "US": 663106, "France": 259452}
test_source2 country values/counts: {"India": 2312565, "US": 1871330, "France": 703378}
test_source3 country values/counts: {"India": 2405000, "US": 1945701, "France": 731615}
Schema differences: {'train_source1': ['entity_id', 'business_name', 'business_address', 'country'], 'train_source2': ['entity_id', 'business_name', 'business_address', 'country'], 'train_source3': ['entity_id', 'business_name', 'business_address', 'country'], 'test_source1': ['entity_id', 'business_name', 'business_address', 'country'], 'test_source2': ['entity_id', 'business_name', 'business_address', 'country'], 'test_source3': ['entity_id', 'business_name', 'business_address', 'country'], 'train_ground_truth': ['sour

## 6. Unicode and data-quality examples
Inspect examples before selecting cleanup rules. Non-ASCII is not itself an error: valid accented French and Indian-language scripts are present. Mojibake-like sequences should be flagged for review, never silently “repaired” by guessing an encoding.

In [15]:
for label,d in SOURCE_AUDIT.items():
    print('\n',label,'non-ASCII counts:',dict(d['non_ascii']),'non-Latin counts in deterministic sample:',dict(d['non_latin_sample_count']))
    print('Whitespace counts:',dict(d['leading_trailing_whitespace']))
    print('Short names/addresses:',d['short_names'][:10],d['short_addresses'][:10])
    print('Long names/addresses (>=300 chars):',d['long_names'][:5],d['long_addresses'][:5])
    print('Unicode examples:',json.dumps(d['unicode_examples'][:8],ensure_ascii=False))


 train_source1 non-ASCII counts: {'name': 0, 'address': 554} non-Latin counts in deterministic sample: {}
Whitespace counts: {'entity_id': 0, 'business_name': 0, 'business_address': 0, 'country': 0}
Short names/addresses: [] []
Long names/addresses (>=300 chars): [] []
Unicode examples: [{"field": "address", "value": "A-212, Malhotra Complex, Gali No. Â 01, Vikas Marg, Shakarpur, Laxmi Nagar, East Delhi, Delhi"}, {"field": "address", "value": "C 12/27 ÂA & C 12/28, Lahangpura Aurangabad, Varanasi, Uttar Pradesh"}, {"field": "address", "value": "Hyderabad, VspâS Bhavana Surovar, Madhavapuri, Hill Flat No.101, First Floor, Pjr Enclave, Ch, Andanagar, Hyderabad, Plot No.277, Telangana"}, {"field": "address", "value": "14/2, Burdwan Road, Sidhartha Apartment, 6Th Floor, Block Â B, Flat No. 6C (4), Kolkata, Kolkata, Howrah, West Bengal"}, {"field": "address", "value": "8-2-293/82/J/B/87/A, Plot Â B, Journalists Colony, Jubilee Hills, Hyderabad, Telangana"}, {"field": "address", "

## 7. Save and review the completed raw audit
This is the explicit checkpoint. Read the saved JSON and printed reports. The notebook does not proceed into the expensive conversion unless `RUN_CONVERSION` is set to `True` after review.

In [16]:
def json_safe(x):
    if isinstance(x,dict): return {str(k):json_safe(v) for k,v in x.items()}
    if isinstance(x,(list,tuple)): return [json_safe(v) for v in x]
    if isinstance(x,(np.integer,np.floating)): return x.item()
    if isinstance(x,collections.Counter): return dict(x)
    return x
AUDIT_REPORT={'sources':SOURCE_AUDIT,'ground_truth':dict(gt),'schemas':SCHEMAS}
AUDIT_PATH=OUTPUT_ROOT/'raw_audit_report.json'
AUDIT_PATH.write_text(json.dumps(json_safe(AUDIT_REPORT),ensure_ascii=False,indent=2),encoding='utf-8')
print('Audit saved to:',AUDIT_PATH)
if not RUN_CONVERSION:
    print('STOP after audit. Review the report. To continue, set RUN_CONVERSION=True in Cell 2, rerun Cells 1–7, then continue.')

Audit saved to: /content/entity_resolution_processed/raw_audit_report.json
STOP after audit. Review the report. To continue, set RUN_CONVERSION=True in Cell 2, rerun Cells 1–7, then continue.


## 8. Preprocessing design and risk decisions
Architecture: verify inputs → audit all raw files in bounded chunks → audit label cardinalities/references → inspect normalized collisions → review audit → explicitly enable chunked Parquet output → validate outputs and write a Source-1-level split. The audit is read-only. No matching, blocking, model, or submission code is included.

| Potential problem | Detection | Handling | Entity-resolution impact |
|---|---|---|---|
| Empty fields and placeholders | Blank/whitespace counts plus separate counts/examples for literal `NA`, `N/A`, `NULL`, `-`, and `unknown` | Keep literal values; missing flags identify only blank/whitespace fields | Avoids deleting valid names such as `NA` and exposes incomplete evidence |
| Duplicate, empty, malformed, or cross-source IDs | Prefix/format checks and SQLite uniqueness across the ID inventory | Report counts; never auto-delete or rewrite IDs | Prevents bad joins and label corruption |
| Zero, one, or many labels; dangling references | Parse comma-separated references; count cardinalities; validate IDs against S2/S3 | Empty string becomes an empty list; retain raw ground truth and report invalid links | Preserves the task's multi-match semantics and valuable true no-match cases |
| Very short/long text | Length and token-count sample quantiles; short/long examples | Retain text and flag for later matching safeguards | Short names cause false positives; long values affect matching cost |
| Unicode, multilingual text, mojibake | Non-ASCII counts/examples and sampled non-Latin characters | Preserve UTF-8 raw text; NFKC is an added view; do not guess repairs or transliterate globally | Retains French accents and Indian scripts without inventing equivalences |
| Normalization collisions | Bounded raw-to-normalized grouping per field | Report groups; never equate a collision to a match | Helps judge recall/precision tradeoffs before blocking |
| Unseen country | Discover values from every source file | Open-set `country_norm` string; no country whitelist | France and later unseen values survive |
| Large files / malformed rows | Chunked strict TSV parser | Write separate Parquet incrementally; malformed rows raise | Bounds memory and prevents silent parse loss |

**Safe additional views:** NFKC, casefolding, whitespace collapse, token lists, sorted tokens, numeric tokens, character/token counts, and blank flags, always alongside raw fields.

**Risky and kept separate:** converting punctuation to spaces, `&` to `and`, sorted-token equality, or country case normalization. These can help candidate generation but can create collisions, so review collision reports and let later matching decide.

**Do not do:** strip numbers, legal suffixes, or location words; deduplicate by similarity; replace raw text with normalized text; transliterate everything; restrict countries; geocode or use external business information.

The raw text, source IDs, countries, empty label meaning, and original token order remain available. Later blocking can use exact normalized strings, token overlaps, sorted tokens, and numbers; later match scoring can compare raw/normalized text. Split by Source 1 ID so one entity's labels cannot cross train/validation. Any learned matcher transformation must fit on train only. Test rows receive the same stateless text normalization; test labels are never used.

## 9. Conservative additional representations
The data audit shows short names, blank addresses in S2/S3, multilingual text, and France in test. The transformations below preserve raw columns, keep all countries, preserve numeric content, and do not remove legal suffixes or location terms. No global transliteration is used: it can erase distinctions and create false collisions. Punctuation/ampersand normalization is an additional view only.

In [17]:
def normalize_text(value):
    if value is None or pd.isna(value): return ''
    s=unicodedata.normalize('NFKC',str(value)).casefold().replace('&',' and ')
    s=''.join(' ' if unicodedata.category(ch).startswith('P') else ch for ch in s)
    return re.sub(r'\s+',' ',s).strip()
def text_representations(value):
    norm=normalize_text(value); tokens=norm.split() if norm else []
    numbers=[t for t in tokens if any(ch.isdigit() for ch in t)]
    return norm,tokens,' '.join(sorted(tokens)),numbers
examples=["O'Reilly's",'ABC & Sons, Ltd.','École primaire Sainte Pierre','東京商事','123 Main Street']
for x in examples: print(repr(x),'→',text_representations(x))

"O'Reilly's" → ('o reilly s', ['o', 'reilly', 's'], 'o reilly s', [])
'ABC & Sons, Ltd.' → ('abc and sons ltd', ['abc', 'and', 'sons', 'ltd'], 'abc and ltd sons', [])
'École primaire Sainte Pierre' → ('école primaire sainte pierre', ['école', 'primaire', 'sainte', 'pierre'], 'pierre primaire sainte école', [])
'東京商事' → ('東京商事', ['東京商事'], '東京商事', [])
'123 Main Street' → ('123 main street', ['123', 'main', 'street'], '123 main street', ['123'])


## 9. Normalization collision audit
Bounded deterministic sample per file/field. This is an audit only; a collision is never treated as a match. High-frequency examples are written to JSON for review.

In [18]:
def collision_audit(path,field,label,limit=250_000):
    groups=collections.defaultdict(collections.Counter); seen=0; row_index=0
    for df in chunks(path,usecols=[field]):
        vals=df[field].astype(str)
        for value in vals:
            if row_index % SAMPLE_EVERY==0 and value.strip():
                norm=normalize_text(value); groups[norm][value]+=1; seen+=1
            row_index+=1
            if seen>=limit: break
        if seen>=limit: break
    ranked=sorted(groups.items(),key=lambda kv:(sum(kv[1].values()),len(kv[1])),reverse=True)
    report={'file':str(path),'field':field,'sampled_nonempty_rows':seen,
       'distinct_raw_in_sample':len({raw for c in groups.values() for raw in c}),
       'distinct_normalized_in_sample':len(groups),
       'normalized_values_with_multiple_raw_forms':sum(len(c)>1 for c in groups.values()),
       'highest_frequency_groups':[{'normalized':n,'frequency':sum(c.values()),'raw_examples':list(c)[:12]} for n,c in ranked[:50]]}
    out=OUTPUT_ROOT/f'{label}_{field}_collision_audit.json'
    out.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
    print(label,field,'sample rows',seen,'distinct raw/norm',report['distinct_raw_in_sample'],report['distinct_normalized_in_sample'],'collision groups',report['normalized_values_with_multiple_raw_forms'])
    return report
COLLISION_REPORTS={}
for label,path in SOURCE_FILES.items():
    for field in ['business_name','business_address']:
        COLLISION_REPORTS[f'{label}_{field}']=collision_audit(path,field,label)
print('Review collision JSON files before enabling conversion.')

train_source1 business_name sample rows 44137 distinct raw/norm 41703 41524 collision groups 176
train_source1 business_address sample rows 44137 distinct raw/norm 44076 44076 collision groups 0
train_source2 business_name sample rows 100693 distinct raw/norm 99437 98394 collision groups 878
train_source2 business_address sample rows 97267 distinct raw/norm 96974 96942 collision groups 32
train_source3 business_name sample rows 105713 distinct raw/norm 104387 103391 collision groups 887
train_source3 business_address sample rows 102224 distinct raw/norm 101976 101956 collision groups 18
test_source1 business_name sample rows 34651 distinct raw/norm 33168 33108 collision groups 60
test_source1 business_address sample rows 34651 distinct raw/norm 34600 34596 collision groups 4
test_source2 business_name sample rows 97746 distinct raw/norm 96623 96015 collision groups 551
test_source2 business_address sample rows 95206 distinct raw/norm 94931 94890 collision groups 41
test_source3 busines

## 10. Audit gate before conversion
Do not enable conversion until the raw audit and collision reports have completed and been reviewed. In particular, decide whether the observed data quality warrants any additional special handling. This notebook intentionally does not perform guessed mojibake repair, deduplication, or transliteration.

In [25]:
if not RUN_CONVERSION:
    raise RuntimeError('Conversion is disabled. Review Cells 4–10 and output reports, then set RUN_CONVERSION=True in Cell 2 and rerun.')
try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except ImportError as e:
    raise RuntimeError('Parquet conversion needs PyArrow. In Colab run: !pip -q install pyarrow') from e
print('Audit gate passed by explicit user setting.')

Audit gate passed by explicit user setting.


## 11. Chunked TSV → Parquet preprocessing
Parquet output goes to a separate directory. Each raw column is preserved verbatim alongside additional representations. Missing flags identify blank/whitespace-only fields; literal tokens such as `NA` are not automatically treated as missing.

In [26]:
def process_source(label,path):
    target=OUTPUT_ROOT/f'{label}.parquet'
    if target.exists(): target.unlink()
    writer=None; total=0
    for df in chunks(path):
        for col in REQUIRED_SOURCE_COLUMNS: df[col]=df[col].astype('string')
        df['business_name_raw']=df.business_name
        df['business_address_raw']=df.business_address
        df['country_raw']=df.country
        for raw,prefix in [('business_name','business_name'),('business_address','business_address')]:
            reps=[text_representations(v) for v in df[raw].astype(str)]
            df[prefix+'_norm']=[x[0] for x in reps]
            df[prefix+'_tokens']=[x[1] for x in reps]
            df[prefix+'_sorted_tokens']=[x[2] for x in reps]
            df[prefix+'_numbers']=[x[3] for x in reps]
        df['country_norm']=[unicodedata.normalize('NFKC',str(v)).strip().casefold() for v in df.country]
        df['name_length']=df.business_name_raw.str.len().astype('int32')
        df['name_token_count']=df.business_name_tokens.map(len).astype('int32')
        df['address_length']=df.business_address_raw.str.len().astype('int32')
        df['address_token_count']=df.business_address_tokens.map(len).astype('int32')
        df['name_missing']=df.business_name_raw.str.strip().eq('')
        df['address_missing']=df.business_address_raw.str.strip().eq('')
        df['country_missing']=df.country_raw.str.strip().eq('')
        table=pa.Table.from_pandas(df,preserve_index=False)
        if writer is None: writer=pq.ParquetWriter(target,table.schema,compression='zstd')
        writer.write_table(table); total+=len(df)
        print(f'{label}: {total:,} rows',end='\r')
    if writer: writer.close()
    print(f'{label}: wrote {total:,} rows to {target}')
    return target,total
PROCESSED={label:process_source(label,path) for label,path in SOURCE_FILES.items()}

train_source1: wrote 2,206,821 rows to /content/entity_resolution_processed/train_source1.parquet
train_source2: wrote 5,034,616 rows to /content/entity_resolution_processed/train_source2.parquet
train_source3: wrote 5,285,603 rows to /content/entity_resolution_processed/train_source3.parquet
test_source1: wrote 1,732,544 rows to /content/entity_resolution_processed/test_source1.parquet
test_source2: wrote 4,887,273 rows to /content/entity_resolution_processed/test_source2.parquet
test_source3: wrote 5,082,316 rows to /content/entity_resolution_processed/test_source3.parquet


## 12. Processed-data validation
Verify row counts and output schema without loading complete Parquet files into memory.

In [27]:
for label,(path,expected_rows) in PROCESSED.items():
    pf=pq.ParquetFile(path)
    assert pf.metadata.num_rows==expected_rows,(label,pf.metadata.num_rows,expected_rows)
    cols=pf.schema_arrow.names
    for col in ['entity_id','business_name','business_address','country','business_name_raw','business_name_norm','business_address_raw','business_address_norm','country_norm']:
        assert col in cols,(label,col)
    print(label,'rows',pf.metadata.num_rows,'columns',len(cols))

train_source1 rows 2206821 columns 23
train_source2 rows 5034616 columns 23
train_source3 rows 5285603 columns 23
test_source1 rows 1732544 columns 23
test_source2 rows 4887273 columns 23
test_source3 rows 5082316 columns 23


## 13. Ground-truth expansion and Source-1-level split
The raw ground truth is retained as Parquet with empty match arrays for zero-match rows. A positive-pair TSV is also written. A deterministic split assigns each Source 1 entity to train or validation; later learned matching features must be fit using only training labels.

In [28]:
gt_out=OUTPUT_ROOT/'train_ground_truth.parquet'
if gt_out.exists(): gt_out.unlink()
gt_writer=None; pair_out=OUTPUT_ROOT/'train_positive_pairs.tsv'
if pair_out.exists(): pair_out.unlink()
with pair_out.open('w',encoding='utf-8',newline='') as f:
    f.write('source1_entity_id\tmatched_entity_id\n')
    for df in chunks(GROUND_TRUTH_FILE):
        df['source1_entity_id']=df.source1_entity_id.astype('string')
        df['matched_entity_ids_raw']=df.matched_entity_ids.astype('string')
        df['matched_entity_ids']=[[] if not str(v).strip() else [x.strip() for x in str(v).split(',') if x.strip()] for v in df.matched_entity_ids_raw]
        table=pa.Table.from_pandas(df,preserve_index=False)
        if gt_writer is None: gt_writer=pq.ParquetWriter(gt_out,table.schema,compression='zstd')
        gt_writer.write_table(table)
        for sid,ids in zip(df.source1_entity_id,df.matched_entity_ids):
            for rid in ids: f.write(f'{sid}\t{rid}\n')
if gt_writer: gt_writer.close()

split_out=OUTPUT_ROOT/'train_source1_split.tsv'
seen=set(); rng=np.random.default_rng(RANDOM_SEED); split_writer=split_out.open('w',encoding='utf-8',newline='')
split_writer.write('source1_entity_id\tsplit\n')
for df in chunks(SOURCE_FILES['train_source1'],usecols=['entity_id']):
    ids=df.entity_id.astype(str).tolist(); n_val=int(round(len(ids)*VALIDATION_FRACTION))
    # Stable hash assignment is row-order-independent and reproducible per ID.
    for sid in ids:
        score=int(hashlib.sha256((str(RANDOM_SEED)+sid).encode('utf-8')).hexdigest()[:16],16)/float(16**16)
        split='validation' if score<VALIDATION_FRACTION else 'train'
        split_writer.write(f'{sid}\t{split}\n')
split_writer.close()
print('Saved',gt_out,pair_out,split_out)

Saved /content/entity_resolution_processed/train_ground_truth.parquet /content/entity_resolution_processed/train_positive_pairs.tsv /content/entity_resolution_processed/train_source1_split.tsv


## 14. Metadata and final summary
All generated outputs are local to `OUTPUT_ROOT`; original TSVs remain unchanged. No test labels, external data, learned vocabularies, or matching decisions are used.

In [29]:
metadata={'data_root':str(DATA_ROOT),'output_root':str(OUTPUT_ROOT),
 'chunk_size':CHUNK_SIZE,'random_seed':RANDOM_SEED,'validation_fraction':VALIDATION_FRACTION,
 'normalization':'Unicode NFKC; casefold; ampersand becomes “and”; punctuation becomes whitespace; whitespace collapsed',
 'external_data_used':False,'matcher_built':False,'audit':json_safe(AUDIT_REPORT),
 'collision_reports':{k:{x:v for x,v in r.items() if x!='highest_frequency_groups'} for k,r in COLLISION_REPORTS.items()}}
(OUTPUT_ROOT/'metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
print('Preprocessing complete. Output files:')
for p in sorted(OUTPUT_ROOT.iterdir()):
    if p.is_file(): print(p.name, f'{p.stat().st_size/1024**2:.1f} MiB')

Preprocessing complete. Output files:
audit_ids.sqlite 1816.4 MiB
metadata.json 0.1 MiB
raw_audit_report.json 0.0 MiB
test_source1.parquet 316.7 MiB
test_source1_business_address_collision_audit.json 0.0 MiB
test_source1_business_name_collision_audit.json 0.0 MiB
test_source2.parquet 962.3 MiB
test_source2_business_address_collision_audit.json 0.0 MiB
test_source2_business_name_collision_audit.json 0.0 MiB
test_source3.parquet 970.5 MiB
test_source3_business_address_collision_audit.json 0.0 MiB
test_source3_business_name_collision_audit.json 0.0 MiB
train_ground_truth.parquet 99.6 MiB
train_positive_pairs.tsv 187.8 MiB
train_source1.parquet 392.9 MiB
train_source1_business_address_collision_audit.json 0.0 MiB
train_source1_business_name_collision_audit.json 0.0 MiB
train_source1_split.tsv 41.3 MiB
train_source2.parquet 957.7 MiB
train_source2_business_address_collision_audit.json 0.0 MiB
train_source2_business_name_collision_audit.json 0.0 MiB
train_source3.parquet 986.1 MiB
train_sour

# Amazon ML Challenge 2026 — Blocking / Candidate Generation

A data-driven, multi-pass S1→(S2∪S3) candidate-generation notebook. It inspects the existing processed Parquet schema before using fields, evaluates rules on the train split only, freezes the rule set before validation, and generates test candidates only in `MODE="TEST"` or `"FULL"`. It never trains a matcher or uses external business information.

In [30]:
import os

print("Current /content:")
print(os.listdir("/content"))

print("\nProcessed folder exists:")
print(os.path.exists("/content/entity_resolution_processed"))

if os.path.exists("/content/entity_resolution_processed"):
    print(os.listdir("/content/entity_resolution_processed"))

Current /content:
['.config', 'entity_resolution_processed', 'drive', '.ipynb_checkpoints', 'dataset', 'sample_data']

Processed folder exists:
True
['test_source1_business_name_collision_audit.json', 'train_positive_pairs.tsv', 'test_source1_business_address_collision_audit.json', 'train_source1_business_address_collision_audit.json', 'train_source1_split.tsv', 'test_source2.parquet', 'metadata.json', 'train_source3_business_address_collision_audit.json', 'train_source1_business_name_collision_audit.json', 'test_source1.parquet', 'train_source3_business_name_collision_audit.json', 'train_source2_business_address_collision_audit.json', 'train_source2.parquet', 'train_source3.parquet', 'test_source2_business_address_collision_audit.json', 'audit_ids.sqlite', 'test_source3_business_name_collision_audit.json', 'train_ground_truth.parquet', 'test_source3_business_address_collision_audit.json', 'test_source2_business_name_collision_audit.json', 'test_source3.parquet', 'train_source2_busines

In [31]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

matches = list(drive_root.rglob("entity_resolution_processed"))

print("Found:")
for x in matches:
    print(x)

Found:


## Competition and engineering constraints

- One Source 1 entity may have zero, one, or many true targets across S2 and S3. Blocking is retrieval, not a one-to-one assignment.
- F0.5 is precision-weighted downstream, but blocking is judged here primarily on measured positive-pair recall subject to candidate volume/runtime constraints.
- Candidate pairs are restricted to S1→S2/S3. Empty keys never block. Country is open-set and is never used as the only key.
- Validation labels are used only after rule selection on the train Source-1 split. Test has no labels and is never used to tune thresholds.
- Only competition-provided processed data is used. No model, business lookup, geocoding, or external business data.
- This notebook reports empirical metrics only when run. No benchmark values are prefilled or claimed.

## Research notes and implementation decisions

| Technique | Established method / source | Decision here | Limitation |
|---|---|---|---|
| Multi-rule blocking | Splink documents union semantics for multiple blocking rules ([docs](https://moj-analytical-services.github.io/splink/topic_guides/blocking/blocking_rules.html)) | Candidate pairs are unioned and provenance retained | Exact keys miss noisy variants |
| Sorted neighbourhood | Record Linkage Toolkit documents sorted-neighbourhood windows and `block_on` partitions ([docs](https://recordlinkage.readthedocs.io/en/latest/ref-index.html)) | Optional local sample experiment only; full-scale pandas MultiIndex is not used | Window cost and memory can be substantial at millions of rows |
| Blocking / filtering families | Papadakis et al., survey ([paper](https://arxiv.org/abs/1905.06167)) | Exact, token, numeric, character n-gram and retrieval families are treated as complementary candidates | No family guarantees recall |
| Q-grams | Survey includes q-gram blocking as a family ([paper](https://jcst.ict.ac.cn/fileup/1000-9001/PDF/2020-4-5-0350.pdf)) | Optional candidate key generation from observed n-gram frequencies | Frequent grams create huge blocks; suppress with measured frequency and safety caps |
| TF-IDF retrieval | Scikit-learn sparse text-vectorization / nearest-neighbour APIs ([TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html), [NearestNeighbors](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NearestNeighbors.html)) | Optional top-K, chunked experiment; never dense all-pairs similarity | Sparse index may still exceed RAM; multilingual tokenization needs evaluation |
| MinHash / LSH | Datasketch API ([documentation](https://ekzhu.com/datasketch/lsh.html)) | Optional, disabled until package/resources are explicitly checked | Approximate retrieval; threshold is not a recall guarantee |
| Phonetic keys | Record-linkage literature commonly uses phonetic encodings; implementations are language-dependent | Not enabled by default; benchmark only after API/license check | English-centric Soundex is inappropriate as a universal multilingual rule |
| Out-of-core execution | DuckDB memory/spill guidance ([OOM guide](https://duckdb.org/docs/current/guides/performance/oom), [memory management](https://duckdb.org/2024/07/09/memory-management)) | DuckDB scans Parquet and materializes bounded joins; configure RAM, threads, temp dir | Spill requires adequate disk; some operators can still exceed limits |

Research sources describe methods, not performance on this challenge. Candidate rule parameters are implementation choices and must be selected from train-split measurements. Dataset-specific findings are generated in the profiling cells.

## Modes

`SCHEMA_ONLY`: schema/resource inventory only. `DEVELOPMENT` (default): rule benchmarking and selection using train Source-1 IDs only. `VALIDATION`: evaluate already-frozen rules on validation IDs. `TEST`: apply frozen rules to test sets and write candidate artifacts. `FULL`: development followed by validation and test, with explicit confirmation gates. Do not use FULL until you have reviewed development results and frozen the rule config.

In [90]:
from pathlib import Path
import os, json, time, math, hashlib, platform, re, sys, warnings, collections
import numpy as np
import pandas as pd
import psutil

PERSISTENT_ROOT = Path(
    "/content/drive/MyDrive/Amazon_ML_Challenge_2026"
)
DATA_ROOT = PERSISTENT_ROOT / "entity_resolution_processed"
OUT = PERSISTENT_ROOT / "entity_resolution_blocking"

MODE = "TEST"  # SCHEMA_ONLY | DEVELOPMENT | VALIDATION | TEST | FULL
SEED = 2026

MAX_BLOCK_ROWS = 20_000
MAX_BLOCK_PAIRS = 2_000_000
MAX_SINGLE_S1_CANDIDATES = 100_000
MAX_TOTAL_CANDIDATES_PER_RULE = 100_000_000
MAX_DF_FRACTION = 0.02
MAX_TOKEN_DF = 100_000
TOKEN_MIN_LENGTH = 2
GRAM_N = 3
GRAM_MIN_LENGTH = 5
TOP_K = 20
THREADS = max(1, min(8, os.cpu_count() or 1))

# Keep DuckDB spill files on Colab's local disk, not Google Drive.
TEMP_DIR = "/content/entity_resolution_blocking_tmp"
Path(TEMP_DIR).mkdir(parents=True, exist_ok=True)

for sub in [
    "cache", "rule_candidates", "metrics", "indexes",
    "validation", "test", "reports", "block_provenance",
]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

if MODE not in {
    "SCHEMA_ONLY", "DEVELOPMENT", "VALIDATION", "TEST", "FULL"
}:
    raise ValueError(f"Unsupported MODE={MODE}")

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Processed data folder not found: {DATA_ROOT}")

if not (OUT / "selected_rules.json").is_file():
    raise FileNotFoundError(
        f"Selected rules not found: {OUT / 'selected_rules.json'}"
    )

print(
    "MODE:", MODE,
    "| DATA_ROOT:", DATA_ROOT,
    "| OUT:", OUT,
    "| CPU:", os.cpu_count(),
    "| RAM_GB:", round(psutil.virtual_memory().total / 2**30, 2),
    "| free_disk_GB:", round(psutil.disk_usage("/content").free / 2**30, 2),
)

MODE: TEST | DATA_ROOT: /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_processed | OUT: /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking | CPU: 2 | RAM_GB: 12.67 | free_disk_GB: 71.28


## Processed-file discovery and schema inspection

In [67]:
EXPECTED={k:DATA_ROOT/f'{k}.parquet' for k in ['train_source1','train_source2','train_source3','test_source1','test_source2','test_source3']}
EXPECTED['train_ground_truth']=DATA_ROOT/'train_ground_truth.parquet'
EXPECTED['train_pairs']=DATA_ROOT/'train_positive_pairs.tsv'
EXPECTED['split']=DATA_ROOT/'train_source1_split.tsv'
for k,p in EXPECTED.items(): print(k, 'FOUND' if p.exists() else 'MISSING', p)
try:
    import duckdb
except ImportError as e: raise RuntimeError('DuckDB is required. In Colab run: !pip -q install duckdb') from e
con=duckdb.connect(str(OUT/'blocking.duckdb'))
con.execute(f"SET threads={THREADS}")
con.execute("SET preserve_insertion_order=false")
con.execute(f"SET temp_directory='{TEMP_DIR}'")
ram_gb=psutil.virtual_memory().available/2**30
con.execute(f"SET memory_limit='{max(1,int(ram_gb*0.55))}GB'")
SCHEMAS={}; ROW_COUNTS={}
for k,p in EXPECTED.items():
    if not p.exists():
        if k in {'train_pairs','split'}: continue
        raise FileNotFoundError(f'Required processed file missing: {p}. Do not rerun preprocessing silently.')
    if p.suffix=='.parquet':
        try:
            SCHEMAS[k]=con.execute(f"DESCRIBE SELECT * FROM read_parquet('{p}')").df()
            ROW_COUNTS[k]=con.execute(f"SELECT count(*) FROM read_parquet('{p}')").fetchone()[0]
        except Exception as e: raise RuntimeError(f'Invalid/unreadable Parquet {p}: {e}') from e
print({k:{'rows':ROW_COUNTS[k],'columns':SCHEMAS[k][['column_name','column_type']].to_dict('records')} for k in SCHEMAS})
SCHEMA_REPORT={k:SCHEMAS[k][['column_name','column_type']].to_dict('records') for k in SCHEMAS}
(OUT/'schema_report.json').write_text(json.dumps(SCHEMA_REPORT,indent=2))
RESOURCE={'cpu_count':os.cpu_count(),'threads':THREADS,'ram_total_bytes':psutil.virtual_memory().total,'ram_available_bytes':psutil.virtual_memory().available,'disk_free_bytes':psutil.disk_usage('/content').free,'duckdb_memory_limit_gb':max(1,int(ram_gb*.55))}
(OUT/'resource_profile.json').write_text(json.dumps(RESOURCE,indent=2))

train_source1 FOUND /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_processed/train_source1.parquet
train_source2 FOUND /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_processed/train_source2.parquet
train_source3 FOUND /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_processed/train_source3.parquet
test_source1 FOUND /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_processed/test_source1.parquet
test_source2 FOUND /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_processed/test_source2.parquet
test_source3 FOUND /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_processed/test_source3.parquet
train_ground_truth FOUND /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_processed/train_ground_truth.parquet
train_pairs FOUND /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_processed/train_positive_pairs.tsv
split FOUND /content/drive/MyDrive/Amazon_ML_Challe

173

## Schema map and training split / labels

The notebook discovers actual column names and types. It prefers normalized fields when available; raw columns are not silently treated as normalized. Token/numeric blocks are skipped with a reason if corresponding usable fields do not exist. Target corpus is train S2+S3, validation S2+S3; test uses test S2+S3.

In [35]:
def cols(table): return set(SCHEMAS[table]['column_name'].tolist())
def first_existing(table,names): return next((x for x in names if x in cols(table)),None)
FIELD_MAP={t:{'id':first_existing(t,['entity_id']),'name':first_existing(t,['business_name_norm','business_name_raw','business_name']),'address':first_existing(t,['business_address_norm','business_address_raw','business_address']),'country':first_existing(t,['country_norm','country']),'name_tokens':first_existing(t,['business_name_tokens']),'address_tokens':first_existing(t,['business_address_tokens']),'name_numbers':first_existing(t,['business_name_numbers']),'address_numbers':first_existing(t,['business_address_numbers']),'name_missing':first_existing(t,['name_missing']),'address_missing':first_existing(t,['address_missing'])} for t in ['train_source1','train_source2','train_source3','test_source1','test_source2','test_source3']}
for t,m in FIELD_MAP.items():
    if not m['id']: raise ValueError(f'{t}: entity_id absent')
    if not m['name'] and not m['address']: raise ValueError(f'{t}: no usable name/address field')
print(json.dumps(FIELD_MAP,indent=2))
if EXPECTED['split'].exists():
    split_df=pd.read_csv(EXPECTED['split'],sep='\t',dtype='string',keep_default_na=False)
    if not {'source1_entity_id','split'}<=set(split_df): raise ValueError('Split manifest schema mismatch')
    if split_df.source1_entity_id.duplicated().any(): raise ValueError('Duplicate S1 in split manifest')
    print('Split counts',split_df.split.value_counts().to_dict())
else: raise FileNotFoundError('train_source1_split.tsv is required; create it in preprocessing first.')
if EXPECTED['train_pairs'].exists():
    pairs=pd.read_csv(EXPECTED['train_pairs'],sep='\t',dtype='string',keep_default_na=False)
    if not {'source1_entity_id','matched_entity_id'}<=set(pairs.columns): raise ValueError(f'Positive pair schema unexpected: {pairs.columns.tolist()}')
    if len(pairs.columns)<2: raise ValueError('Positive pair file requires source1_entity_id and matched_entity_id')
    pairs=pairs.iloc[:,:2]; pairs.columns=['source1_entity_id','matched_entity_id']
else: raise FileNotFoundError('train_positive_pairs.tsv required; no labels will be invented.')
if pairs.duplicated().any(): pairs=pairs.drop_duplicates()
if not pairs.matched_entity_id.str.startswith(('S2-','S3-')).all(): raise ValueError('Positive target IDs must be S2/S3')
print('Positive pair count',len(pairs))
train_ids=set(split_df.loc[split_df.split.eq('train'),'source1_entity_id'].astype(str))
valid_ids=set(split_df.loc[split_df.split.eq('validation'),'source1_entity_id'].astype(str))
if train_ids & valid_ids: raise ValueError('Train/validation S1 overlap')


{
  "train_source1": {
    "id": "entity_id",
    "name": "business_name_norm",
    "address": "business_address_norm",
    "country": "country_norm",
    "name_tokens": "business_name_tokens",
    "address_tokens": "business_address_tokens",
    "name_numbers": "business_name_numbers",
    "address_numbers": "business_address_numbers",
    "name_missing": "name_missing",
    "address_missing": "address_missing"
  },
  "train_source2": {
    "id": "entity_id",
    "name": "business_name_norm",
    "address": "business_address_norm",
    "country": "country_norm",
    "name_tokens": "business_name_tokens",
    "address_tokens": "business_address_tokens",
    "name_numbers": "business_name_numbers",
    "address_numbers": "business_address_numbers",
    "name_missing": "name_missing",
    "address_missing": "address_missing"
  },
  "train_source3": {
    "id": "entity_id",
    "name": "business_name_norm",
    "address": "business_address_norm",
    "country": "country_norm",
    "name_t

## Frequency profile and safe rule registry

In [68]:
# Profile values are derived from target corpora. No stopword list is assumed.

PROFILE = {}

for t in ["train_source2", "train_source3"]:
    for field in [
        FIELD_MAP[t]["name"],
        FIELD_MAP[t]["address"],
        FIELD_MAP[t]["country"],
    ]:
        if field:
            PROFILE[(t, field)] = con.execute(
                f"""
                SELECT
                    count(*) AS row_count,
                    count(DISTINCT {field}) AS distinct_values,
                    count(*) FILTER (
                        WHERE {field} IS NULL
                           OR trim(CAST({field} AS VARCHAR)) = ''
                    ) AS empty_count
                FROM read_parquet('{EXPECTED[t]}')
                """
            ).fetchone()

print("Field profiles:", PROFILE)

RULES = []


def add_rule(
    rule_id, family, left_expr, right_expr, configuration,
    available=True, reason=""
):
    RULES.append({
        "rule_id": rule_id,
        "family": family,
        "left_expr": left_expr,
        "right_expr": right_expr,
        "configuration": configuration,
        "available": available,
        "skip_reason": reason,
    })


def qid(s):
    return '"' + s.replace('"', '""') + '"'


def expr_for(t, field):
    return qid(FIELD_MAP[t][field]) if FIELD_MAP[t][field] else None


# Broad independent exact rules. Empty keys are excluded in the runner.
for field in ["name", "address"]:
    if all(
        FIELD_MAP[t][field]
        for t in ["train_source1", "train_source2", "train_source3"]
    ):
        for rawtarget in ["train_source2", "train_source3"]:
            add_rule(
                "exact_" + field + "_" + rawtarget,
                "exact",
                expr_for("train_source1", field),
                expr_for(rawtarget, field),
                {"field": field, "target": rawtarget},
            )
    else:
        add_rule(
            "exact_" + field,
            "exact",
            None,
            None,
            {},
            False,
            f"{field} representation absent in one or more source schemas",
        )

for field in ["name", "address"]:
    if all(
        FIELD_MAP[t][field] and FIELD_MAP[t]["country"]
        for t in ["train_source1", "train_source2", "train_source3"]
    ):
        add_rule(
            field + "_country",
            "composite",
            None,
            None,
            {
                "field": field,
                "country": True,
                "targets": ["train_source2", "train_source3"],
            },
        )

# Token and numeric signatures are considered only when these representations exist.
for token_field in ["name_tokens", "address_tokens", "address_numbers"]:
    available = all(
        FIELD_MAP[t][token_field]
        for t in ["train_source1", "train_source2", "train_source3"]
    )
    add_rule(
        "shared_" + token_field,
        "token" if "tokens" in token_field else "numeric",
        None,
        None,
        {
            "field": token_field,
            "max_document_frequency": MAX_TOKEN_DF,
        },
        available,
        "Required representation absent" if not available else "",
    )

# Additional families are documented/skipped unless explicitly implemented and enabled.
for rid, fam, why in [
    (
        "char_prefix_suffix",
        "character_key",
        "Not enabled in this baseline; frequency/safety profiling is needed first",
    ),
    (
        "qgram_inverted",
        "qgram",
        "Optional experiment; requires an external-memory inverted index and per-gram block caps",
    ),
    (
        "sorted_neighbourhood",
        "sorted_neighbourhood",
        "Requires bounded sampling; full-corpus pandas indexing risks memory exhaustion",
    ),
    (
        "phonetic",
        "phonetic",
        "Disabled: multilingual corpus and no verified multilingual phonetic representation in schema",
    ),
    (
        "transliteration_repair",
        "transliteration",
        "Disabled unless a precomputed column exists; do not guess mojibake repairs",
    ),
    (
        "tfidf_topk",
        "tfidf",
        "Optional sparse top-K experiment; not run by default due to RAM/runtime variability",
    ),
    (
        "minhash_lsh",
        "lsh",
        "Optional approximate retrieval; package/API/license/resource checks are required",
    ),
]:
    add_rule(rid, fam, None, None, {}, False, why)

print(
    pd.DataFrame([
        {k: v for k, v in r.items() if k not in ("left_expr", "right_expr")}
        for r in RULES
    ]).to_string(index=False)
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Field profiles: {('train_source2', 'business_name_norm'): (5034616, 4037471, 0), ('train_source2', 'business_address_norm'): (5034616, 4286097, 168967), ('train_source2', 'country_norm'): (5034616, 2, 0), ('train_source3', 'business_name_norm'): (5285603, 4293938, 0), ('train_source3', 'business_address_norm'): (5285603, 4616119, 175916), ('train_source3', 'country_norm'): (5285603, 2, 0)}
                    rule_id               family                                                                        configuration  available                                                                                  skip_reason
   exact_name_train_source2                exact                                         {'field': 'name', 'target': 'train_source2'}       True                                                                                             
   exact_name_train_source3                exact                                         {'field': 'name', 'target': 'train_source3

## Candidate rule runner and block-size guard

Rules are built as source-restricted equijoins (never a Cartesian join). Before a join, key multiplicities are profiled. Oversized keys are recorded with estimated pair products and suppressed; their affected gold positives are measured where possible. Candidate artifacts checkpoint by rule/config fingerprint.

In [69]:
def source_sql(table): return f"read_parquet('{EXPECTED[table]}')"
def split_filter(alias,split_name):
    ids=train_ids if split_name=='train' else valid_ids
    temp=f"ids_{split_name}"
    con.register(temp,pd.DataFrame({'entity_id':list(ids)}))
    return f" JOIN {temp} sf ON CAST({alias}.{qid(FIELD_MAP['train_source1']['id'])} AS VARCHAR)=sf.entity_id "
def make_rule_sql(rule,split_name):
    targets=rule['configuration'].get('targets',[rule['configuration'].get('target')])
    pieces=[]
    for target in targets:
        if target is None: continue
        a='a'; b='b'; src=target
        left_id=FIELD_MAP['train_source1']['id']; right_id=FIELD_MAP[src]['id']
        if rule['family']=='exact':
            le=rule['left_expr']; re_=rule['right_expr']
            key_cond=f"CAST({a}.{le} AS VARCHAR)=CAST({b}.{re_} AS VARCHAR) AND trim(CAST({a}.{le} AS VARCHAR))<>''"
        elif rule['family']=='composite':
            f=rule['configuration']['field']; le=expr_for('train_source1',f); re_=expr_for(src,f); lc=expr_for('train_source1','country'); rc=expr_for(src,'country')
            key_cond=f"CAST({a}.{le} AS VARCHAR)=CAST({b}.{re_} AS VARCHAR) AND CAST({a}.{lc} AS VARCHAR)=CAST({b}.{rc} AS VARCHAR) AND trim(CAST({a}.{le} AS VARCHAR))<>'' AND trim(CAST({a}.{lc} AS VARCHAR))<>''"
        else: continue
        source='S2' if src.endswith('2') else 'S3'
        filt=split_filter(a,split_name)
        pieces.append(f"SELECT CAST({a}.{qid(left_id)} AS VARCHAR) source1_entity_id, CAST({b}.{qid(right_id)} AS VARCHAR) candidate_entity_id, '{source}' candidate_source FROM {source_sql('train_source1')} {a} {filt} JOIN {source_sql(src)} {b} ON {key_cond}")
    return ' UNION ALL '.join(pieces) if pieces else None

def oversized_keys(rule,split_name='train'):
    # Exact-key guard. Composite rules add country; the aggregate uses all S1 IDs in that partition only.
    if rule['family'] not in {'exact','composite'}: return 0,0,None
    fields=[rule['configuration']['field']]
    if rule['family']=='composite': fields+=['country']
    expression=" || chr(31) || ".join([f"coalesce(CAST({expr_for('train_source1',f)} AS VARCHAR),'')" for f in fields])
    targets=rule['configuration'].get('targets',[rule['configuration'].get('target')]); total_unsafe=0; unsafe_pairs=0
    for target in targets:
        if not target: continue
        rexpr=" || chr(31) || ".join([f"coalesce(CAST({expr_for(target,f)} AS VARCHAR),'')" for f in fields])
        # temp IDs restrict left side to chosen split.
        filt=split_filter('a',split_name)
        query=f"WITH l AS (SELECT {expression} k,count(*) n FROM {source_sql('train_source1')} a {filt} WHERE trim(CAST({expr_for('train_source1',fields[0])} AS VARCHAR))<>'' GROUP BY 1), r AS (SELECT {rexpr} k,count(*) n FROM {source_sql(target)} GROUP BY 1) SELECT count(*),coalesce(sum(l.n*r.n),0) FROM l JOIN r USING(k) WHERE l.n+r.n>{MAX_BLOCK_ROWS} OR l.n*r.n>{MAX_BLOCK_PAIRS}"
        count,pairs_=con.execute(query).fetchone(); total_unsafe+=count; unsafe_pairs+=pairs_
    return total_unsafe,unsafe_pairs,None

def rule_artifact(rule,split_name):
    fp=hashlib.sha256(json.dumps({'rule':rule['rule_id'],'config':rule['configuration'],'split':split_name,'source_rows':ROW_COUNTS},sort_keys=True).encode()).hexdigest()[:16]
    return OUT/'rule_candidates'/f'{split_name}_{rule["rule_id"]}_{fp}.parquet'

def run_rule(rule,split_name='train'):
    started=time.time(); artifact=rule_artifact(rule,split_name)
    if not rule['available']:
        return {'rule_id':rule['rule_id'],'skipped':True,'skip_reason':rule['skip_reason'],'runtime_seconds':0.0,'artifact':None}
    # Current SQL implementation enables exact normalized keys and field+country; token/numeric families are listed explicitly pending list-index implementation in this notebook version.
    if rule['family'] not in {'exact','composite'}:
        return {'rule_id':rule['rule_id'],'skipped':True,'skip_reason':'Representation discovered, but safe SQL unnest/index implementation is not enabled in this notebook version. No silent fallback.','runtime_seconds':time.time()-started,'artifact':None}
    sql=make_rule_sql(rule,split_name)
    if not sql: return {'rule_id':rule['rule_id'],'skipped':True,'skip_reason':'No valid target SQL','runtime_seconds':0.0,'artifact':None}
    unsafe_count,unsafe_pairs,_=oversized_keys(rule,split_name)
    if unsafe_count:
        return {'rule_id':rule['rule_id'],'skipped':True,'skip_reason':f'Unsafe oversized blocks detected ({unsafe_count} keys; estimated {unsafe_pairs} pairs). This baseline fails closed rather than materialize them; refine the rule before enabling.', 'runtime_seconds':time.time()-started,'oversized_block_count':unsafe_count,'unsafe_pair_estimate':unsafe_pairs,'artifact':None}
    # Safe blocks pass to materialization. Oversized blocks fail closed; they are not silently dropped.
    if artifact.exists():
        try: con.execute(f"SELECT count(*) FROM read_parquet('{artifact}')").fetchone()
        except Exception: artifact.unlink()
    if not artifact.exists():
        tmp=artifact.with_suffix('.tmp.parquet')
        try:
            con.execute(f"COPY ({sql}) TO '{tmp}' (FORMAT PARQUET, COMPRESSION ZSTD)")
            tmp.replace(artifact)
        except Exception as e:
            if tmp.exists(): tmp.unlink()
            raise RuntimeError(f'Rule {rule["rule_id"]} failed. Artifact not marked complete: {e}') from e
    count=con.execute(f"SELECT count(*) FROM read_parquet('{artifact}')").fetchone()[0]
    return {'rule_id':rule['rule_id'],'skipped':False,'runtime_seconds':time.time()-started,'candidate_pairs':count,'oversized_block_count':unsafe_count,'unsafe_pair_estimate':unsafe_pairs,'artifact':str(artifact)}

## Development benchmark, rule metrics, and train-only selection

In [48]:
def truth_table(split_name):
    ids=train_ids if split_name=='train' else valid_ids
    return pairs[pairs.source1_entity_id.isin(ids)].copy()
def metrics_for_artifact(result,split_name):
    truth=truth_table(split_name); total=len(truth)
    if result.get('skipped') or not result.get('artifact'):
        return {'rule_id':result['rule_id'],'skipped':True,'skip_reason':result.get('skip_reason',''),'candidate_pairs':0,'pair_recall':0.0,'s2_pair_recall':0.0,'s3_pair_recall':0.0,'runtime_seconds':result.get('runtime_seconds',0)}
    path=result['artifact']; con.execute(f"CREATE OR REPLACE TEMP VIEW cands AS SELECT DISTINCT * FROM read_parquet('{path}')")
    hits=con.execute("SELECT p.matched_entity_id FROM (SELECT source1_entity_id,matched_entity_id FROM read_csv_auto(?) ) p JOIN cands c ON p.source1_entity_id=c.source1_entity_id AND p.matched_entity_id=c.candidate_entity_id",[str(EXPECTED['train_pairs'])]).df() if False else None
    # Register only split-scoped positives (small relative to corpus); compare in DuckDB.
    con.register('truth_df',truth.rename(columns={'matched_entity_id':'candidate_entity_id'}))
    hitdf=con.execute('SELECT t.source1_entity_id,t.candidate_entity_id,c.candidate_source FROM truth_df t JOIN cands c USING(source1_entity_id,candidate_entity_id)').df()
    n_hit=len(hitdf); s2=int(hitdf.candidate_source.eq('S2').sum()); s3=int(hitdf.candidate_source.eq('S3').sum())
    per=con.execute('SELECT count(*) n FROM cands').fetchone()[0]
    dist=con.execute('SELECT count(*) n FROM cands GROUP BY source1_entity_id').df().n if per else pd.Series(dtype=int)
    non_single=truth.groupby('source1_entity_id').size(); captured=hitdf.groupby('source1_entity_id').candidate_entity_id.nunique()
    return {'rule_id':result['rule_id'],'family':next((r['family'] for r in RULES if r['rule_id']==result['rule_id']),''),'configuration':next((r['configuration'] for r in RULES if r['rule_id']==result['rule_id']),{}),'candidate_pairs':int(per),'unique_candidate_pairs':int(per),'pair_recall':n_hit/total if total else None,'s2_pair_recall':s2/max(1,int(truth.matched_entity_id.str.startswith('S2-').sum())),'s3_pair_recall':s3/max(1,int(truth.matched_entity_id.str.startswith('S3-').sum())),'any_match_entity_coverage':float(captured.reindex(non_single.index,fill_value=0).gt(0).mean()) if len(non_single) else None,'complete_match_entity_coverage':float((captured.reindex(non_single.index,fill_value=0)==non_single).mean()) if len(non_single) else None,'zero_candidate_entities':int(len((train_ids if split_name=='train' else valid_ids))-len(dist)),'mean_candidates_per_s1':float(dist.mean()) if len(dist) else 0.0,'median_candidates_per_s1':float(dist.median()) if len(dist) else 0.0,'p90_candidates_per_s1':float(dist.quantile(.9)) if len(dist) else 0.0,'p95_candidates_per_s1':float(dist.quantile(.95)) if len(dist) else 0.0,'p99_candidates_per_s1':float(dist.quantile(.99)) if len(dist) else 0.0,'max_candidates_per_s1':int(dist.max()) if len(dist) else 0,'reduction_ratio':1-per/(len(train_ids if split_name=='train' else valid_ids)*(ROW_COUNTS['train_source2']+ROW_COUNTS['train_source3'])),'runtime_seconds':result.get('runtime_seconds',0),'oversized_block_count':result.get('oversized_block_count',0),'unsafe_pair_estimate':result.get('unsafe_pair_estimate',0),'skipped':False,'skip_reason':''}

if MODE in {'DEVELOPMENT','FULL'}:
    DEV_RESULTS=[]; DEV_METRICS=[]
    for rule in RULES:
        result=run_rule(rule,'train'); DEV_RESULTS.append(result)
        metric=metrics_for_artifact(result,'train'); DEV_METRICS.append(metric)
        print(rule['rule_id'],metric)
    dev_df=pd.DataFrame(DEV_METRICS); dev_df.to_parquet(OUT/'metrics/blocking_rule_metrics_dev.parquet',index=False)
    eligible=dev_df[(~dev_df.skipped.fillna(True)) & (dev_df.candidate_pairs<=MAX_TOTAL_CANDIDATES_PER_RULE)]
    # Transparent configurable heuristic: maximize recall; ties prefer fewer candidates then shorter measured runtime.
    selected=eligible.sort_values(['pair_recall','candidate_pairs','runtime_seconds'],ascending=[False,True,True]).head(5)
    SELECTED_RULE_IDS=selected.rule_id.tolist()
    if not SELECTED_RULE_IDS: raise RuntimeError('No benchmarked rule is eligible. Inspect skipped/errors; do not generate a fabricated candidate set.')
    selected_rules=[r for r in RULES if r['rule_id'] in SELECTED_RULE_IDS]
    (OUT/'selected_rules.json').write_text(json.dumps({'rule_ids':SELECTED_RULE_IDS,'selection_basis':'train split only; descending pair recall, ascending candidates, ascending runtime','metrics':selected.to_dict('records')},indent=2,default=str))
    print('Selected on train only:',SELECTED_RULE_IDS)
else:
    config=OUT/'selected_rules.json'
    if not config.exists(): raise FileNotFoundError('Frozen selected_rules.json missing. Run DEVELOPMENT and review first.')
    SELECTED_RULE_IDS=json.loads(config.read_text())['rule_ids']; selected_rules=[r for r in RULES if r['rule_id'] in SELECTED_RULE_IDS]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

name_country {'rule_id': 'name_country', 'family': 'composite', 'configuration': {'field': 'name', 'country': True, 'targets': ['train_source2', 'train_source3']}, 'candidate_pairs': 18278106, 'unique_candidate_pairs': 18278106, 'pair_recall': 0.21621001445368862, 's2_pair_recall': 0.21202894440851, 's3_pair_recall': 0.2201263605516791, 'any_match_entity_coverage': 0.5430458076674515, 'complete_match_entity_coverage': 0.02748923275772462, 'zero_candidate_entities': 552063, 'mean_candidates_per_s1': 13.819087286256371, 'median_candidates_per_s1': 2.0, 'p90_candidates_per_s1': 35.0, 'p95_candidates_per_s1': 77.0, 'p99_candidates_per_s1': 198.0, 'max_candidates_per_s1': 1039, 'reduction_ratio': 0.9999990552810596, 'runtime_seconds': 33.26716899871826, 'oversized_block_count': 0, 'unsafe_pair_estimate': 0, 'skipped': False, 'skip_reason': ''}


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

address_country {'rule_id': 'address_country', 'family': 'composite', 'configuration': {'field': 'address', 'country': True, 'targets': ['train_source2', 'train_source3']}, 'candidate_pairs': 652855, 'unique_candidate_pairs': 652855, 'pair_recall': 0.08260624401198274, 's2_pair_recall': 0.12476206255994239, 's3_pair_recall': 0.04311951604666787, 'any_match_entity_coverage': 0.23641158207282406, 'complete_match_entity_coverage': 0.006127046679893164, 'zero_candidate_entities': 1422422, 'mean_candidates_per_s1': 1.4433731583508729, 'median_candidates_per_s1': 1.0, 'p90_candidates_per_s1': 2.0, 'p95_candidates_per_s1': 3.0, 'p99_candidates_per_s1': 6.0, 'max_candidates_per_s1': 20, 'reduction_ratio': 0.9999999662566524, 'runtime_seconds': 41.71499991416931, 'oversized_block_count': 0, 'unsafe_pair_estimate': 0, 'skipped': False, 'skip_reason': ''}
shared_name_tokens {'rule_id': 'shared_name_tokens', 'skipped': True, 'skip_reason': 'Representation discovered, but safe SQL unnest/index impl

## Multi-pass union, marginal contributions, and validation

In [116]:
def union_artifacts(results,split_name):
    paths=[r['artifact'] for r in results if not r.get('skipped') and r.get('artifact')]
    if not paths: raise RuntimeError('No candidate artifacts to union')
    union=OUT/('validation' if split_name=='validation' else 'rule_candidates')/f'union_{split_name}.parquet'
    # UNION ALL then group-by deduplicates pairs and preserves rule provenance.
    selects=[]
    for r in results:
        if r.get('artifact') and not r.get('skipped'):
            selects.append(f"SELECT source1_entity_id,candidate_entity_id,candidate_source,'{r['rule_id']}' rule_name FROM read_parquet('{r['artifact']}')")
    sql=' UNION ALL '.join(selects)
    con.execute(f"COPY (SELECT source1_entity_id,candidate_entity_id,any_value(candidate_source) candidate_source,count(DISTINCT rule_name) rule_count,string_agg(DISTINCT rule_name, ',') rules_triggered FROM ({sql}) GROUP BY 1,2) TO '{union}' (FORMAT PARQUET, COMPRESSION ZSTD)")
    return union

def evaluate_union(path,split_name):
    truth=truth_table(split_name); con.register('union_truth',truth.rename(columns={'matched_entity_id':'candidate_entity_id'}))
    v=f"read_parquet('{path}')"; total=len(truth)
    hits=con.execute(f'SELECT t.source1_entity_id,t.candidate_entity_id,c.candidate_source FROM union_truth t JOIN {v} c USING(source1_entity_id,candidate_entity_id)').df()
    count=con.execute(f'SELECT count(*) FROM {v}').fetchone()[0]
    per=con.execute(f'SELECT count(*) n FROM {v} GROUP BY source1_entity_id').df().n
    by_entity=truth.groupby('source1_entity_id').size(); got=hits.groupby('source1_entity_id').candidate_entity_id.nunique()
    return {'split':split_name,'candidate_pairs':int(count),'pair_recall':len(hits)/total if total else None,'s2_pair_recall':int(hits.candidate_source.eq('S2').sum())/max(1,int(truth.matched_entity_id.str.startswith('S2-').sum())),'s3_pair_recall':int(hits.candidate_source.eq('S3').sum())/max(1,int(truth.matched_entity_id.str.startswith('S3-').sum())),'any_match_entity_coverage':float(got.reindex(by_entity.index,fill_value=0).gt(0).mean()) if len(by_entity) else None,'complete_match_entity_coverage':float((got.reindex(by_entity.index,fill_value=0)==by_entity).mean()) if len(by_entity) else None,'candidate_distribution':{'mean':float(per.mean()) if len(per) else 0,'median':float(per.median()) if len(per) else 0,'p90':float(per.quantile(.9)) if len(per) else 0,'p95':float(per.quantile(.95)) if len(per) else 0,'p99':float(per.quantile(.99)) if len(per) else 0,'max':int(per.max()) if len(per) else 0},'zero_candidate_entities':len((train_ids if split_name=='train' else valid_ids))-len(per),'reduction_ratio':1-count/(len(train_ids if split_name=='train' else valid_ids)*(ROW_COUNTS['train_source2']+ROW_COUNTS['train_source3']))}

if MODE in {'VALIDATION','FULL'}:
    val_results=[]
    for rule in selected_rules:
        val_results.append(run_rule(rule,'validation'))
    val_metrics=[metrics_for_artifact(r,'validation') for r in val_results]
    pd.DataFrame(val_metrics).to_parquet(OUT/'metrics/blocking_rule_metrics_validation.parquet',index=False)
    val_union=union_artifacts(val_results,'validation')
    VALIDATION_METRICS=evaluate_union(val_union,'validation')
    (OUT/'validation/selected_block_metrics.json').write_text(json.dumps(VALIDATION_METRICS,indent=2))
    print('Frozen selected blocker validation:',VALIDATION_METRICS)


In [117]:
train_union = union_artifacts(train_results, "train")
print("Train union created:", train_union)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

OutOfMemoryException: Out of Memory Error: could not allocate block of size 256.0 KiB (4.6 GiB/4.6 GiB used)

Possible solutions:
* Reducing the number of threads (SET threads=X)
* Disabling insertion-order preservation (SET preserve_insertion_order=false)
* Increasing the memory limit (SET memory_limit='...GB')

See also https://duckdb.org/docs/stable/guides/performance/how_to_tune_workloads

## Test candidates — explicit mode only

In [99]:
if MODE in {"TEST", "FULL"}:
    import shutil

    frozen = OUT / "selected_rules.json"
    if not frozen.exists():
        raise FileNotFoundError(
            "No frozen selected rules. Run development first."
        )

    if MODE == "FULL" and not (
        OUT / "validation/selected_block_metrics.json"
    ).exists():
        raise RuntimeError(
            "FULL mode requires completed validation review before test generation."
        )

    # The baseline SQL runner supports exact and field+country rules.
    test_results = []

    for rule in selected_rules:
        if rule["family"] not in {"exact", "composite"}:
            test_results.append({
                "rule_id": rule["rule_id"],
                "skipped": True,
                "skip_reason": "test SQL implementation unavailable for this family",
                "artifact": None,
            })
            continue

        targets = []

        for src in ["test_source2", "test_source3"]:
            if rule["family"] == "exact":
                field = rule["configuration"]["field"]
                left_field = FIELD_MAP["test_source1"][field]
                right_field = FIELD_MAP[src][field]

                if not left_field or not right_field:
                    continue

                condition = (
                    f"CAST(a.{qid(left_field)} AS VARCHAR) = "
                    f"CAST(b.{qid(right_field)} AS VARCHAR) "
                    f"AND trim(CAST(a.{qid(left_field)} AS VARCHAR)) <> ''"
                )
            else:
                field = rule["configuration"]["field"]
                left_field = FIELD_MAP["test_source1"][field]
                right_field = FIELD_MAP[src][field]
                left_country = FIELD_MAP["test_source1"]["country"]
                right_country = FIELD_MAP[src]["country"]

                if not all([
                    left_field, right_field, left_country, right_country
                ]):
                    continue

                condition = (
                    f"CAST(a.{qid(left_field)} AS VARCHAR) = "
                    f"CAST(b.{qid(right_field)} AS VARCHAR) "
                    f"AND CAST(a.{qid(left_country)} AS VARCHAR) = "
                    f"CAST(b.{qid(right_country)} AS VARCHAR) "
                    f"AND trim(CAST(a.{qid(left_field)} AS VARCHAR)) <> '' "
                    f"AND trim(CAST(a.{qid(left_country)} AS VARCHAR)) <> ''"
                )

            source_name = "S2" if src.endswith("2") else "S3"
            targets.append(
                f"""
                SELECT
                    CAST(a.entity_id AS VARCHAR) AS source1_entity_id,
                    CAST(b.entity_id AS VARCHAR) AS candidate_entity_id,
                    '{source_name}' AS candidate_source,
                    '{rule["rule_id"]}' AS rule_name
                FROM read_parquet('{EXPECTED["test_source1"]}') AS a
                JOIN read_parquet('{EXPECTED[src]}') AS b
                    ON {condition}
                """
            )

        if not targets:
            test_results.append({
                "rule_id": rule["rule_id"],
                "skipped": True,
                "skip_reason": "No compatible test-source fields were available",
                "artifact": None,
            })
            continue

        rule_path = OUT / "test" / f"{rule['rule_id']}.parquet"
        rule_path.parent.mkdir(parents=True, exist_ok=True)
        temp_path = rule_path.with_suffix(".tmp.parquet")

        union_sql = " UNION ALL ".join(targets)
        con.execute(f"""
            COPY ({union_sql})
            TO '{temp_path}'
            (FORMAT PARQUET, COMPRESSION ZSTD)
        """)
        temp_path.replace(rule_path)

        test_results.append({
            "rule_id": rule["rule_id"],
            "skipped": False,
            "artifact": str(rule_path),
        })

    # Union the per-rule artifacts.
    test_union = union_artifacts(test_results, "test")

    OUT_TEST = OUT / "candidate_pairs_test.parquet"
    OUT_TEST.parent.mkdir(parents=True, exist_ok=True)

    if OUT_TEST.exists():
        OUT_TEST.unlink()

    shutil.copyfile(test_union, OUT_TEST)

    if not OUT_TEST.is_file() or OUT_TEST.stat().st_size == 0:
        raise RuntimeError(
            f"Test candidate Parquet was not created correctly: {OUT_TEST}"
        )


    test_count = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{OUT_TEST}')"
    ).fetchone()[0]

    print("Test candidate pairs:", f"{test_count:,}")
    print("Test candidate Parquet:", OUT_TEST)

    # Export one source1 row per entity, including entities with no candidates.
    # Hash partitions bound the aggregation memory required by each query.
    PARTITIONS = 512
    tsv = OUT / "candidate_pairs.tsv"
    source1_path = EXPECTED["test_source1"]

    if tsv.exists():
        tsv.unlink()

    for bucket in range(PARTITIONS):
        shard = OUT / f"_candidate_pairs_tsv_part_{bucket:04d}.tsv"

        con.execute(f"""
            COPY (
                WITH s1 AS (
                    SELECT CAST(entity_id AS VARCHAR) AS source1_entity_id
                    FROM read_parquet('{source1_path}')
                    WHERE hash(CAST(entity_id AS VARCHAR)) % {PARTITIONS} = {bucket}
                )
                SELECT
                    s1.source1_entity_id,
                    coalesce(
                        string_agg(
                            DISTINCT c.candidate_entity_id,
                            ',' ORDER BY c.candidate_entity_id
                        ),
                        ''
                    ) AS candidate_entity_ids
                FROM s1
                LEFT JOIN read_parquet('{OUT_TEST}') AS c
                    ON s1.source1_entity_id = c.source1_entity_id
                GROUP BY s1.source1_entity_id
            )
            TO '{shard}' (HEADER, DELIMITER '\\t')
        """)

        # Append each partition while keeping only one header.
        with shard.open("rb") as src_file:
            src_file.readline()
            with tsv.open("ab") as dst_file:
                shutil.copyfileobj(src_file, dst_file, length=1024 * 1024)

        shard.unlink()

    print(
        "TSV:",
        tsv,
        "size:",
        f"{tsv.stat().st_size:,} bytes",
    )

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

TSV: /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking/candidate_pairs.tsv size: 170,571,012 bytes


## Candidate provenance, subgroup reporting, and integrity checks

In [101]:
import shutil

tsv = OUT / "candidate_pairs.tsv"
fixed_tsv = OUT / "candidate_pairs_with_header.tsv"

with fixed_tsv.open("wb") as dst:
    dst.write(b"source1_entity_id\tcandidate_entity_ids\n")
    with tsv.open("rb") as src:
        shutil.copyfileobj(src, dst, length=1024 * 1024)

fixed_tsv.replace(tsv)

out_rows = con.execute(
    f"SELECT COUNT(*) FROM read_csv('{tsv}', delim='\\t', header=true)"
).fetchone()[0]

print("TSV rows:", out_rows)
print("Expected source1 rows:", ROW_COUNTS["test_source1"])

TSV rows: 1732544
Expected source1 rows: 1732544


In [102]:
# Persist final configs/reports and expose skipped/rejected techniques rather than implying they ran.
(OUT/'skipped_rules.json').write_text(json.dumps([{'rule_id':r['rule_id'],'family':r['family'],'reason':r['skip_reason'] or 'not implemented/enabled in baseline'} for r in RULES if not r['available'] or r['family'] not in {'exact','composite'}],indent=2))
(OUT/'rejected_rules.json').write_text(json.dumps([],indent=2))
(OUT/'config.json').write_text(json.dumps({'mode':MODE,'seed':SEED,'selected_rule_ids':SELECTED_RULE_IDS,'safety':{'MAX_BLOCK_ROWS':MAX_BLOCK_ROWS,'MAX_BLOCK_PAIRS':MAX_BLOCK_PAIRS,'MAX_SINGLE_S1_CANDIDATES':MAX_SINGLE_S1_CANDIDATES,'MAX_TOTAL_CANDIDATES_PER_RULE':MAX_TOTAL_CANDIDATES_PER_RULE},'data_root':str(DATA_ROOT)},indent=2))
if MODE in {'DEVELOPMENT','FULL'}:
    print('Development rule metrics:')
    print(dev_df[['rule_id','candidate_pairs','pair_recall','runtime_seconds','skipped']].to_string(index=False))
if MODE in {'TEST','FULL'}:
    # Candidate integrity checks: all targets exist in test S2/S3; no S1 IDs; pair uniqueness.
    dup=con.execute(f"SELECT count(*) FROM (SELECT source1_entity_id,candidate_entity_id,count(*) n FROM read_parquet('{OUT_TEST}') GROUP BY 1,2 HAVING n>1)").fetchone()[0]
    bad=con.execute(f"SELECT count(*) FROM read_parquet('{OUT_TEST}') WHERE candidate_entity_id NOT LIKE 'S2-%' AND candidate_entity_id NOT LIKE 'S3-%'").fetchone()[0]
    n_s1=ROW_COUNTS['test_source1']; out_rows=con.execute(f"SELECT count(*) FROM read_csv('{tsv}',delim='\t',header=true)").fetchone()[0]
    if dup or bad or n_s1!=out_rows: raise RuntimeError(f'Integrity failure duplicates={dup}, invalid targets={bad}, S1 rows={n_s1}, TSV rows={out_rows}')
    print('Integrity OK:',{'duplicate_pairs':dup,'invalid_target_ids':bad,'test_s1_rows':n_s1,'tsv_rows':out_rows})
summary={'mode':MODE,'resource_profile':RESOURCE,'schema_report':SCHEMA_REPORT,'selected_rule_ids':SELECTED_RULE_IDS,'skipped_rule_count':sum((not r['available']) or r['family'] not in {'exact','composite'} for r in RULES)}
if 'VALIDATION_METRICS' in globals(): summary['validation']=VALIDATION_METRICS
(OUT/'final_blocking_summary.json').write_text(json.dumps(summary,indent=2,default=str))
print('Outputs at',OUT)
print('No matcher was trained and no matching decisions were made.')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Integrity OK: {'duplicate_pairs': 0, 'invalid_target_ids': 0, 'test_s1_rows': 1732544, 'tsv_rows': 1732544}
Outputs at /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking
No matcher was trained and no matching decisions were made.


In [103]:
from google.colab import drive
from pathlib import Path
import shutil

# 1. Mount Google Drive
drive.mount("/content/drive")

# 2. Persistent destination in Google Drive
DRIVE_ROOT = Path("/content/drive/MyDrive/Amazon_ML_Challenge_2026")

# Create destination
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# 3. Folders you want to permanently save
folders = [
    Path("/content/entity_resolution_processed"),
    Path("/content/entity_resolution_blocking"),
]

# 4. Copy them to Google Drive
for src in folders:
    if not src.exists():
        print(f"❌ Not found: {src}")
        continue

    dest = DRIVE_ROOT / src.name

    print(f"\nCopying:")
    print(f"FROM: {src}")
    print(f"TO:   {dest}")

    shutil.copytree(
        src,
        dest,
        dirs_exist_ok=True
    )

    print(f"✅ Saved: {dest}")

print("\n==============================")
print("✅ BACKUP COMPLETE")
print("==============================")

print("\nGoogle Drive contents:")
for item in DRIVE_ROOT.iterdir():
    print(" -", item)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Copying:
FROM: /content/entity_resolution_processed
TO:   /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_processed
✅ Saved: /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_processed

Copying:
FROM: /content/entity_resolution_blocking
TO:   /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking
✅ Saved: /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking

✅ BACKUP COMPLETE

Google Drive contents:
 - /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_processed
 - /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking
 - /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_features


In [104]:
from pathlib import Path
import duckdb

# Local Colab disk is much faster than Google Drive for DuckDB temporary files.
DUCKDB_TMP = Path("/content/duckdb_tmp")
DUCKDB_TMP.mkdir(parents=True, exist_ok=True)

# IMPORTANT:
# Close the old connection BEFORE configuring temp_directory.
try:
    con.close()
    print("Closed previous DuckDB connection.")
except Exception:
    pass

# Reconnect with all settings applied BEFORE any query uses the connection.
DB = Path(
    "/content/drive/MyDrive/Amazon_ML_Challenge_2026/"
    "entity_resolution_blocking/blocking.duckdb"
)

con = duckdb.connect(
    str(DB),
    read_only=False,
    config={
        "threads": "1",
        "memory_limit": "3GB",
        "temp_directory": str(DUCKDB_TMP),
        "max_temp_directory_size": "40GB",
        "preserve_insertion_order": "false",
    },
)

print("✅ Fresh DuckDB connection created")
print("Database:", DB)
print("Temp directory:", DUCKDB_TMP)
print("Memory limit: 3GB")
print("Temp disk limit: 40GB")

Closed previous DuckDB connection.
✅ Fresh DuckDB connection created
Database: /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking/blocking.duckdb
Temp directory: /content/duckdb_tmp
Memory limit: 3GB
Temp disk limit: 40GB


In [118]:
def union_artifacts(results, split_name, partition_count=512):
    import shutil
    from pathlib import Path

    # Keep temporary files on Colab local storage, not Google Drive.
    work_dir = Path(f"/content/er_union_work/{split_name}")
    raw_dir = work_dir / "raw"
    part_dir = work_dir / "parts"
    raw_dir.mkdir(parents=True, exist_ok=True)
    part_dir.mkdir(parents=True, exist_ok=True)

    free_gb = shutil.disk_usage("/content").free / (1024 ** 3)
    print(f"Local free disk: {free_gb:.1f} GB")
    if free_gb < 5:
        raise RuntimeError(
            "Less than 5 GB free on /content. Free up local disk space "
            "before building the union."
        )

    usable = [
        r for r in results
        if not r.get("skipped") and r.get("artifact")
    ]
    if not usable:
        raise RuntimeError(f"No usable candidate artifacts for {split_name}")

    # Make smaller raw files first; this step does not globally group candidates.
    raw_files = []
    for i, result in enumerate(usable):
        raw_path = raw_dir / f"rule_{i:04d}.parquet"
        artifact = str(result["artifact"]).replace("'", "''")
        rule_id = result["rule_id"].replace("'", "''")

        con.execute(
            f"""
            COPY (
                SELECT
                    source1_entity_id,
                    candidate_entity_id,
                    candidate_source,
                    '{rule_id}' AS rule_name
                FROM read_parquet('{artifact}')
            )
            TO '{raw_path}'
            (FORMAT PARQUET, COMPRESSION ZSTD)
            """
        )
        raw_files.append(str(raw_path))

    # Lower DuckDB's working pressure so it spills earlier when needed.
    con.execute("SET threads=2")
    con.execute("SET preserve_insertion_order=false")
    con.execute("SET memory_limit='1.5GB'")

    raw_list = "[" + ",".join(
        "'" + p.replace("'", "''") + "'" for p in raw_files
    ) + "]"

    final_path = OUT / (
        "validation/union_validation.parquet"
        if split_name == "validation"
        else f"rule_candidates/union_{split_name}.parquet"
    )
    final_path.parent.mkdir(parents=True, exist_ok=True)

    part_files = []
    for part_id in range(partition_count):
        part_path = part_dir / f"part_{part_id:04d}.parquet"
        part_files.append(str(part_path))

        key_hash = (
            "hash(source1_entity_id || chr(31) || candidate_entity_id)"
        )

        con.execute(
            f"""
            COPY (
                SELECT
                    source1_entity_id,
                    candidate_entity_id,
                    any_value(candidate_source) AS candidate_source,
                    count(DISTINCT rule_name) AS rule_count,
                    string_agg(DISTINCT rule_name, ',') AS rules_triggered
                FROM read_parquet({raw_list})
                WHERE {key_hash} % {partition_count} = {part_id}
                GROUP BY source1_entity_id, candidate_entity_id
            )
            TO '{part_path}'
            (FORMAT PARQUET, COMPRESSION ZSTD)
            """
        )

        if (part_id + 1) % 32 == 0:
            print(f"Deduplicated {part_id + 1}/{partition_count} partitions")

    # This final pass only concatenates already-deduplicated partitions.
    temp_final = final_path.with_suffix(".tmp.parquet")
    parts_glob = str(part_dir / "part_*.parquet").replace("'", "''")

    con.execute(
        f"""
        COPY (
            SELECT *
            FROM read_parquet('{parts_glob}')
        )
        TO '{temp_final}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
        """
    )
    temp_final.replace(final_path)

    # Verify uniqueness and report the resulting pair count.
    duplicate_pairs = con.execute(
        f"""
        SELECT count(*)
        FROM (
            SELECT source1_entity_id, candidate_entity_id, count(*) AS n
            FROM read_parquet('{final_path}')
            GROUP BY source1_entity_id, candidate_entity_id
            HAVING n > 1
        )
        """
    ).fetchone()[0]

    if duplicate_pairs:
        raise RuntimeError(
            f"Union still contains {duplicate_pairs} duplicate pairs."
        )

    pair_count = con.execute(
        f"SELECT count(*) FROM read_parquet('{final_path}')"
    ).fetchone()[0]

    print("Union saved:", final_path)
    print("Unique candidate pairs:", f"{pair_count:,}")
    return final_path

In [120]:
train_union = union_artifacts(train_results, "train")

Local free disk: 69.4 GB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Deduplicated 512/512 partitions


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Union saved: /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking/rule_candidates/union_train.parquet
Unique candidate pairs: 18,903,790


In [3]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, time, hashlib, shutil, platform, subprocess, collections, re, unicodedata, math
import numpy as np
import pandas as pd
import psutil

PERSISTENT_ROOT=Path('/content/drive/MyDrive/Amazon_ML_Challenge_2026')
PROCESSED_ROOT=PERSISTENT_ROOT/'entity_resolution_processed'
BLOCKING_ROOT=PERSISTENT_ROOT/'entity_resolution_blocking'
PERSISTENT_FEATURE_ROOT=PERSISTENT_ROOT/'entity_resolution_features'
RUN_VERSION='v1'
RUN_ROOT=PERSISTENT_FEATURE_ROOT/RUN_VERSION
LOCAL_WORK_ROOT=Path('/content/amazon_ml_feature_engineering_work')
RANDOM_SEED=2026
SHARD_ROWS=50_000
RESUME=True
OVERWRITE=False
WRITE_SUCCESS=True
ENABLE_TFIDF=False  # Optional: benchmark/cap training corpus before enabling.
if RUN_ROOT.exists() and (RUN_ROOT/'SUCCESS.json').exists() and not OVERWRITE:
    raise FileExistsError(f'{RUN_ROOT} already has SUCCESS.json. Choose a new RUN_VERSION or explicitly set OVERWRITE=True.')
if RUN_ROOT.exists() and any(RUN_ROOT.iterdir()) and not RESUME and not OVERWRITE:
    raise FileExistsError(f'{RUN_ROOT} already contains files. Set RESUME=True or choose a new version.')
if OVERWRITE and RUN_ROOT.exists():
    raise RuntimeError('OVERWRITE is intentionally not automated. Choose a new RUN_VERSION to protect completed artifacts.')
for d in ['metadata','fit_artifacts','train','validation','test','diagnostics','checkpoints','manifests']:
    (RUN_ROOT/d).mkdir(parents=True,exist_ok=True)
LOCAL_WORK_ROOT.mkdir(parents=True,exist_ok=True)
print('[OK] Drive:',PERSISTENT_ROOT,'local scratch:',LOCAL_WORK_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[OK] Drive: /content/drive/MyDrive/Amazon_ML_Challenge_2026 local scratch: /content/amazon_ml_feature_engineering_work


In [57]:
%pip install -q rapidfuzz duckdb pyarrow scikit-learn psutil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.4 MB/s eta 0:00:00


In [4]:
try:
    import duckdb, pyarrow as pa, pyarrow.parquet as pq
    from rapidfuzz import fuzz
except ImportError as e:
    raise RuntimeError('Install required packages in Colab: !pip -q install duckdb pyarrow rapidfuzz') from e
try:
    import sklearn
except ImportError as e:
    raise RuntimeError('scikit-learn is required for optional TF-IDF features.') from e
RESOURCE_PROFILE={'python':sys.version,'platform':platform.platform(),'cpu_count':os.cpu_count(),'ram_total_bytes':psutil.virtual_memory().total,'ram_available_bytes':psutil.virtual_memory().available,'local_disk_free_bytes':psutil.disk_usage('/content').free,'drive_disk_free_bytes':psutil.disk_usage(str(PERSISTENT_ROOT)).free,'packages':{'duckdb':duckdb.__version__,'pyarrow':pa.__version__,'pandas':pd.__version__,'numpy':np.__version__,'rapidfuzz':__import__('rapidfuzz').__version__,'scikit_learn':sklearn.__version__}}
print(json.dumps(RESOURCE_PROFILE,indent=2))
( RUN_ROOT/'metadata/resource_profile.json').write_text(json.dumps(RESOURCE_PROFILE,indent=2))

{
  "python": "3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "cpu_count": 2,
  "ram_total_bytes": 13605830656,
  "ram_available_bytes": 11593801728,
  "local_disk_free_bytes": 74300084224,
  "drive_disk_free_bytes": 70585077760,
  "packages": {
    "duckdb": "1.3.2",
    "pyarrow": "23.0.1",
    "pandas": "2.2.3",
    "numpy": "2.1.3",
    "rapidfuzz": "3.14.6",
    "scikit_learn": "1.6.1"
  }
}


461

In [5]:
for root in [PERSISTENT_ROOT,PROCESSED_ROOT,BLOCKING_ROOT]:
    if not root.exists(): raise FileNotFoundError(f'Required Drive path is missing: {root}')
def inventory(root):
    out=[]
    for p in sorted(root.rglob('*')):
        if p.is_file():
            st=p.stat(); rec={'path':str(p),'relative_path':str(p.relative_to(root)),'size_bytes':st.st_size,'modified_time':st.st_mtime}
            if p.suffix.lower()=='.parquet':
                try:
                    pf=pq.ParquetFile(p); rec['rows']=pf.metadata.num_rows; rec['columns']=[{'name':f.name,'type':str(f.type)} for f in pf.schema_arrow]
                except Exception as e: rec['parquet_error']=str(e)
            out.append(rec)
    return out
PROCESSED_INVENTORY=inventory(PROCESSED_ROOT); BLOCKING_INVENTORY=inventory(BLOCKING_ROOT)
print('PROCESSED ARTIFACTS'); print(json.dumps(PROCESSED_INVENTORY,indent=2)[:30000])
print('BLOCKING ARTIFACTS'); print(json.dumps(BLOCKING_INVENTORY,indent=2)[:30000])
(RUN_ROOT/'metadata/processed_inventory.json').write_text(json.dumps(PROCESSED_INVENTORY,indent=2))
(RUN_ROOT/'metadata/blocking_inventory.json').write_text(json.dumps(BLOCKING_INVENTORY,indent=2))

PROCESSED ARTIFACTS
[
  {
    "path": "/content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_processed/audit_ids.sqlite",
    "relative_path": "audit_ids.sqlite",
    "size_bytes": 1904640000,
    "modified_time": 1790323937.0
  },
  {
    "path": "/content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_processed/metadata.json",
    "relative_path": "metadata.json",
    "size_bytes": 54075,
    "modified_time": 1790326185.0
  },
  {
    "path": "/content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_processed/raw_audit_report.json",
    "relative_path": "raw_audit_report.json",
    "size_bytes": 45580,
    "modified_time": 1790324100.0
  },
  {
    "path": "/content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_processed/test_source1.parquet",
    "relative_path": "test_source1.parquet",
    "size_bytes": 332083307,
    "modified_time": 1790325416.0,
    "rows": 1732544,
    "columns": [
      {
        "name": "entity_id",
        "type": "s

19439

In [6]:
def unique_file(root, basenames, suffix='.parquet'):
    hits=[p for p in root.rglob('*') if p.is_file() and p.suffix.lower()==suffix and p.name in basenames]
    if len(hits)!=1: raise RuntimeError(f'Expected exactly one {basenames} under {root}; found {[str(x) for x in hits]}')
    return hits[0]
RECORD_PATHS={}
for s in ['train_source1','train_source2','train_source3','test_source1','test_source2','test_source3']:
    RECORD_PATHS[s]=unique_file(PROCESSED_ROOT,[s+'.parquet'])

def parquet_schema(path):
    pf=pq.ParquetFile(path)
    return {'rows':pf.metadata.num_rows,'columns':[{'name':f.name,'type':str(f.type)} for f in pf.schema_arrow]}
SCHEMA_INVENTORY={k:parquet_schema(p) for k,p in RECORD_PATHS.items()}
for k,v in SCHEMA_INVENTORY.items(): print(k,v)
(RUN_ROOT/'metadata/schema_inventory.json').write_text(json.dumps(SCHEMA_INVENTORY,indent=2))
con=duckdb.connect(str(LOCAL_WORK_ROOT/'features.duckdb'))
con.execute('SET preserve_insertion_order=false')
con.execute(f"SET threads={max(1,min(8,os.cpu_count() or 1))}")
mem_gb=max(1,int(psutil.virtual_memory().available/2**30*.5))
con.execute(f"SET memory_limit='{mem_gb}GB'")


train_source1 {'rows': 2206821, 'columns': [{'name': 'entity_id', 'type': 'string'}, {'name': 'business_name', 'type': 'string'}, {'name': 'business_address', 'type': 'string'}, {'name': 'country', 'type': 'string'}, {'name': 'business_name_raw', 'type': 'string'}, {'name': 'business_address_raw', 'type': 'string'}, {'name': 'country_raw', 'type': 'string'}, {'name': 'business_name_norm', 'type': 'string'}, {'name': 'business_name_tokens', 'type': 'list<element: string>'}, {'name': 'business_name_sorted_tokens', 'type': 'string'}, {'name': 'business_name_numbers', 'type': 'list<element: string>'}, {'name': 'business_address_norm', 'type': 'string'}, {'name': 'business_address_tokens', 'type': 'list<element: string>'}, {'name': 'business_address_sorted_tokens', 'type': 'string'}, {'name': 'business_address_numbers', 'type': 'list<element: string>'}, {'name': 'country_norm', 'type': 'string'}, {'name': 'name_length', 'type': 'int32'}, {'name': 'name_token_count', 'type': 'int32'}, {'name

In [7]:
from pathlib import Path

BLOCKING_ROOT = Path(
    "/content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking"
)

print("Blocking root:", BLOCKING_ROOT)
print("Exists:", BLOCKING_ROOT.exists())

print("\nAll files:")
for p in sorted(BLOCKING_ROOT.rglob("*")):
    if p.is_file():
        print(p)

Blocking root: /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking
Exists: True

All files:
/content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking/blocking.duckdb
/content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking/candidate_pairs.tsv
/content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking/candidate_pairs_test.parquet
/content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking/config.json
/content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking/final_blocking_summary.json
/content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking/metrics/blocking_rule_metrics_dev.parquet
/content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking/metrics/blocking_rule_metrics_validation.parquet
/content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking/rejected_rules.json
/content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blo

In [9]:
from pathlib import Path

BLOCKING_ROOT = Path(
    "/content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking"
)

CANDIDATE_PATHS = {
    "train": BLOCKING_ROOT / "rule_candidates/union_train.parquet",
    "validation": BLOCKING_ROOT / "validation/union_validation.parquet",
    "test": BLOCKING_ROOT / "candidate_pairs_test.parquet",
}

for split, path in CANDIDATE_PATHS.items():
    if not path.is_file():
        raise FileNotFoundError(f"{split} file missing: {path}")
    if path.stat().st_size == 0:
        raise RuntimeError(f"{split} file is empty: {path}")
    print(f"{split}: {path} ({path.stat().st_size:,} bytes)")

train: /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking/rule_candidates/union_train.parquet (217,362,997 bytes)
validation: /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking/validation/union_validation.parquet (26,813,531 bytes)
test: /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking/candidate_pairs_test.parquet (130,939,988 bytes)


In [10]:
for split, path in CANDIDATE_PATHS.items():
    path = Path(path)
    print(f"{split}: {path}")
    if path.exists():
        print(f"  size: {path.stat().st_size:,} bytes")
    else:
        print("  MISSING")

train: /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking/rule_candidates/union_train.parquet
  size: 217,362,997 bytes
validation: /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking/validation/union_validation.parquet
  size: 26,813,531 bytes
test: /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking/candidate_pairs_test.parquet
  size: 130,939,988 bytes


In [11]:
# Inspect candidate-like artifacts
candidate_files = [
    Path(item["path"])
    for item in BLOCKING_INVENTORY
    if Path(item["path"]).suffix.lower() in {".parquet", ".tsv", ".csv"}
    and any(
        word in Path(item["path"]).name.lower()
        for word in ("candidate", "union", "pair")
    )
]

print("Candidate-like files:")
for path in candidate_files:
    print(path)

# Final candidate artifacts, one per split
TRAIN_CANDIDATE_PATH = BLOCKING_ROOT / "rule_candidates/union_train.parquet"
VALIDATION_CANDIDATE_PATH = BLOCKING_ROOT / "validation/union_validation.parquet"
TEST_CANDIDATE_PATH = BLOCKING_ROOT / "candidate_pairs_test.parquet"

CANDIDATE_PATHS = {
    "train": Path(TRAIN_CANDIDATE_PATH),
    "validation": Path(VALIDATION_CANDIDATE_PATH),
    "test": Path(TEST_CANDIDATE_PATH),
}

# Verify each artifact and display its schema
for split, path in CANDIDATE_PATHS.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"{split} final candidates missing: {path}\n"
            "Generate the final candidate artifact for this split, then rerun this cell."
        )

    schema = (
        parquet_schema(path)
        if path.suffix.lower() == ".parquet"
        else "delimited text"
    )
    print(f"{split}: {path}\n  {schema}")

Candidate-like files:
/content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking/candidate_pairs.tsv
/content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking/candidate_pairs_test.parquet
/content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking/rule_candidates/tmp_union_train.parquet
/content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking/rule_candidates/union_test.parquet
/content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking/rule_candidates/union_train.parquet
/content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking/validation/union_validation.parquet
train: /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_blocking/rule_candidates/union_train.parquet
  {'rows': 18903790, 'columns': [{'name': 'source1_entity_id', 'type': 'string'}, {'name': 'candidate_entity_id', 'type': 'string'}, {'name': 'candidate_source', 'type': 'string'}, {'name': 'rule_count', 'type': 'i

In [12]:
def names(path):
    if path.suffix.lower()=='.parquet': return pq.ParquetFile(path).schema_arrow.names
    return pd.read_csv(path,sep='\t' if path.suffix.lower()=='.tsv' else ',',nrows=0).columns.tolist()

def choose(columns, options, label, required=False):
    found=next((x for x in options if x in columns),None)
    if required and found is None: raise KeyError(f'{label}: none of {options} found in {columns}')
    return found
RECORD_MAP={}
for k,p in RECORD_PATHS.items():
    c=names(p)
    RECORD_MAP[k]={'id':choose(c,['entity_id','record_id'],'entity id',True),
        'name_raw':choose(c,['business_name_raw','business_name'],'raw name'),
        'name_norm':choose(c,['business_name_norm'],'normalized name'),
        'name_tokens':choose(c,['business_name_tokens'],'name tokens'),
        'name_sorted':choose(c,['business_name_sorted_tokens'],'sorted name tokens'),
        'name_numbers':choose(c,['business_name_numbers'],'name numbers'),
        'address_raw':choose(c,['business_address_raw','business_address'],'raw address'),
        'address_norm':choose(c,['business_address_norm'],'normalized address'),
        'address_tokens':choose(c,['business_address_tokens'],'address tokens'),
        'address_sorted':choose(c,['business_address_sorted_tokens'],'sorted address tokens'),
        'address_numbers':choose(c,['business_address_numbers'],'address numbers'),
        'country':choose(c,['country_raw','country'],'country'),
        'country_norm':choose(c,['country_norm'],'normalized country'),
        'name_missing':choose(c,['name_missing'],'name missing flag'),
        'address_missing':choose(c,['address_missing'],'address missing flag'),
        'country_missing':choose(c,['country_missing'],'country missing flag')}
print(json.dumps(RECORD_MAP,indent=2))
# Candidate schema auto-map; list-style one-row-per-S1 submissions are exploded in canonicalize_candidates.
CANDIDATE_MAP={}
for split,p in CANDIDATE_PATHS.items():
    c=names(p)
    CANDIDATE_MAP[split]={'s1':choose(c,['source1_entity_id','entity_id'],'S1 candidate key',True),
        'candidate':choose(c,['candidate_entity_id','matched_entity_id'],'candidate key'),
        'candidate_list':choose(c,['candidate_entity_ids','matched_entity_ids'],'candidate list'),
        'source':choose(c,['candidate_source','source'],'candidate source'),
        'provenance':choose(c,['rules_triggered','blocking_rules','provenance'],'blocking provenance')}
    if not CANDIDATE_MAP[split]['candidate'] and not CANDIDATE_MAP[split]['candidate_list']:
        raise KeyError(f'{split} candidate file needs candidate_entity_id or candidate_entity_ids')
print('Candidate mappings:',CANDIDATE_MAP)

{
  "train_source1": {
    "id": "entity_id",
    "name_raw": "business_name_raw",
    "name_norm": "business_name_norm",
    "name_tokens": "business_name_tokens",
    "name_sorted": "business_name_sorted_tokens",
    "name_numbers": "business_name_numbers",
    "address_raw": "business_address_raw",
    "address_norm": "business_address_norm",
    "address_tokens": "business_address_tokens",
    "address_sorted": "business_address_sorted_tokens",
    "address_numbers": "business_address_numbers",
    "country": "country_raw",
    "country_norm": "country_norm",
    "name_missing": "name_missing",
    "address_missing": "address_missing",
    "country_missing": "country_missing"
  },
  "train_source2": {
    "id": "entity_id",
    "name_raw": "business_name_raw",
    "name_norm": "business_name_norm",
    "name_tokens": "business_name_tokens",
    "name_sorted": "business_name_sorted_tokens",
    "name_numbers": "business_name_numbers",
    "address_raw": "business_address_raw",
    "

In [13]:
SPLIT_PATH=unique_file(PROCESSED_ROOT,['train_source1_split.tsv'],suffix='.tsv')
POSITIVE_PATHS=[p for p in PROCESSED_ROOT.rglob('*') if p.is_file() and p.name in {'train_positive_pairs.tsv','train_ground_truth.parquet'}]
if not POSITIVE_PATHS: raise FileNotFoundError('No provided training positive-pair/ground-truth artifact found.')
split_df=pd.read_csv(SPLIT_PATH,sep='\t',dtype='string',keep_default_na=False)
if not {'source1_entity_id','split'}<=set(split_df.columns): raise ValueError(f'Unexpected split schema {split_df.columns.tolist()}')
if split_df.source1_entity_id.duplicated().any(): raise ValueError('Duplicate Source-1 ID in split file')
if set(split_df.split.unique())-{'train','validation'}: raise ValueError('Split file contains unknown split labels')
train_s1=set(split_df.loc[split_df.split=='train','source1_entity_id'].astype(str)); val_s1=set(split_df.loc[split_df.split=='validation','source1_entity_id'].astype(str))
if train_s1 & val_s1: raise ValueError('S1 train/validation leakage')
positive_path=next((p for p in POSITIVE_PATHS if p.name=='train_positive_pairs.tsv'),None)
if positive_path:
    positives=pd.read_csv(positive_path,sep='\t',dtype='string',keep_default_na=False)
    if not {'source1_entity_id','matched_entity_id'}<=set(positives.columns): raise ValueError(f'Unexpected positive-pair schema: {positives.columns.tolist()}')
else:
    gp=next(p for p in POSITIVE_PATHS if p.suffix=='.parquet')
    positives=con.execute(f"SELECT source1_entity_id, unnest(matched_entity_ids) AS matched_entity_id FROM read_parquet('{gp}')").df()
positives=positives[['source1_entity_id','matched_entity_id']].drop_duplicates()
if not positives.matched_entity_id.astype(str).str.startswith(('S2-','S3-')).all(): raise ValueError('Ground truth target IDs must be S2/S3')
print('S1 train/validation:',len(train_s1),len(val_s1),'positive pairs:',len(positives))
(RUN_ROOT/'diagnostics/candidate_label_audit.parquet').parent.mkdir(exist_ok=True,parents=True)

S1 train/validation: 1874734 332087 positive pairs: 7638365


In [14]:
def canonicalize_candidates(split,path,mapping):
    out=LOCAL_WORK_ROOT/f'candidates_{split}.parquet'
    if out.exists(): out.unlink()
    cols=mapping; quoted=lambda x: '"'+x.replace('"','""')+'"'
    if path.suffix.lower()=='.parquet': src=f"read_parquet('{path}')"
    else:
        delim='\t' if path.suffix.lower()=='.tsv' else ','
        src=f"read_csv('{path}',delim='{delim}',header=true,all_varchar=true)"
    s1=quoted(cols['s1']); cand=quoted(cols['candidate']) if cols['candidate'] else None
    if cand:
        src_expr=f"SELECT CAST({s1} AS VARCHAR) source1_entity_id, CAST({cand} AS VARCHAR) candidate_entity_id"
        if cols['source']: src_expr+=f", CAST({quoted(cols['source'])} AS VARCHAR) candidate_source"
        else: src_expr+=", CASE WHEN CAST("+quoted(cols['candidate'])+" AS VARCHAR) LIKE 'S2-%' THEN 'S2' WHEN CAST("+quoted(cols['candidate'])+" AS VARCHAR) LIKE 'S3-%' THEN 'S3' ELSE NULL END candidate_source"
        if cols['provenance']: src_expr+=f", CAST({quoted(cols['provenance'])} AS VARCHAR) rules_triggered"
        else: src_expr+=", NULL::VARCHAR rules_triggered"
        src_expr+=f" FROM {src}"
    else:
        ls=quoted(cols['candidate_list'])
        src_expr=f"SELECT CAST({s1} AS VARCHAR) source1_entity_id, trim(x) candidate_entity_id, CASE WHEN trim(x) LIKE 'S2-%' THEN 'S2' WHEN trim(x) LIKE 'S3-%' THEN 'S3' ELSE NULL END candidate_source, NULL::VARCHAR rules_triggered FROM {src}, UNNEST(string_split(CAST({ls} AS VARCHAR), ',')) u(x) WHERE trim(x)<>''"
    con.execute(f"COPY (SELECT source1_entity_id,candidate_entity_id, CASE WHEN candidate_entity_id LIKE 'S2-%' THEN 'S2' WHEN candidate_entity_id LIKE 'S3-%' THEN 'S3' ELSE NULL END AS candidate_source, string_agg(DISTINCT rules_triggered, ',') FILTER (WHERE rules_triggered IS NOT NULL AND rules_triggered<>'') AS rules_triggered FROM ({src_expr}) WHERE source1_entity_id IS NOT NULL AND candidate_entity_id IS NOT NULL GROUP BY source1_entity_id,candidate_entity_id) TO '{out}' (FORMAT PARQUET, COMPRESSION ZSTD)")
    return out
CANONICAL={s:canonicalize_candidates(s,p,CANDIDATE_MAP[s]) for s,p in CANDIDATE_PATHS.items()}
for split,p in CANONICAL.items():
    count=con.execute(f"SELECT count(*) FROM read_parquet('{p}')").fetchone()[0]
    dup=con.execute(f"SELECT count(*) FROM (SELECT source1_entity_id,candidate_entity_id,count(*) n FROM read_parquet('{p}') GROUP BY 1,2 HAVING n>1)").fetchone()[0]
    invalid=con.execute(f"SELECT count(*) FROM read_parquet('{p}') WHERE candidate_source NOT IN ('S2','S3') OR candidate_entity_id LIKE 'S1-%'").fetchone()[0]
    if dup or invalid: raise RuntimeError(f'{split} candidate integrity failure: duplicates={dup}, invalid={invalid}')
    print(split,'pairs',count,'duplicates',dup,'invalid',invalid)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

train pairs 18903790 duplicates 0 invalid 0
validation pairs 3348934 duplicates 0 invalid 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

test pairs 11408860 duplicates 0 invalid 0


In [16]:
# Candidate absence from the supplied positive table is a negative only
# within the candidate universe. Positive truth pairs missing from the
# final candidates are counted separately as blocking losses.

for split in ["train", "validation"]:
    cand = CANONICAL[split]
    split_ids = train_s1 if split == "train" else val_s1

    truth = positives[
        positives["source1_entity_id"].astype(str).isin(split_ids)
    ][["source1_entity_id", "matched_entity_id"]].drop_duplicates()

    con.register("truth_pairs_df", truth)

    label_path = RUN_ROOT / f"diagnostics/candidate_labels_{split}.parquet"
    label_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        con.execute(f"""
            COPY (
                SELECT
                    c.*,
                    CASE
                        WHEN p.matched_entity_id IS NOT NULL THEN 1
                        ELSE 0
                    END::TINYINT AS "label",
                    (p.matched_entity_id IS NOT NULL)
                        AS candidate_present_in_ground_truth,
                    CASE
                        WHEN p.matched_entity_id IS NULL THEN NULL
                        WHEN p.matched_entity_id LIKE 'S2-%' THEN 'S2'
                        ELSE 'S3'
                    END AS truth_source
                FROM read_parquet('{cand}') AS c
                LEFT JOIN truth_pairs_df AS p
                    ON c.source1_entity_id = p.source1_entity_id
                    AND c.candidate_entity_id = p.matched_entity_id
            )
            TO '{label_path}'
            (FORMAT PARQUET, COMPRESSION ZSTD)
        """)

        cand_truth = con.execute(f"""
            SELECT COUNT(*)
            FROM read_parquet('{label_path}')
            WHERE "label" = 1
        """).fetchone()[0]

        cand_pairs = con.execute(
            f"SELECT COUNT(*) FROM read_parquet('{cand}')"
        ).fetchone()[0]

        print(
            split,
            "candidate pairs:", cand_pairs,
            "candidate positives:", cand_truth,
            "candidate negatives:", cand_pairs - cand_truth,
        )
        print("Labeled candidates:", label_path)

    finally:
        con.unregister("truth_pairs_df")

print(
    "[WARN] Blocking losses are counted separately in the next cell. "
    "Only candidate pairs absent from supplied positives are labeled negative."
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

train candidate pairs: 18903790 candidate positives: 1855032 candidate negatives: 17048758
Labeled candidates: /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_features/v1/diagnostics/candidate_labels_train.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

validation candidate pairs: 3348934 candidate positives: 329001 candidate negatives: 3019933
Labeled candidates: /content/drive/MyDrive/Amazon_ML_Challenge_2026/entity_resolution_features/v1/diagnostics/candidate_labels_validation.parquet
[WARN] Blocking losses are counted separately in the next cell. Only candidate pairs absent from supplied positives are labeled negative.


In [17]:
BLOCKING_COVERAGE={}
for split in ['train','validation']:
    ids=train_s1 if split=='train' else val_s1
    truth=positives[positives.source1_entity_id.astype(str).isin(ids)].copy()
    cand=CANONICAL[split]
    con.register('truth_cov_df',truth)
    total=len(truth)
    captured=con.execute(f"SELECT count(*) FROM truth_cov_df t JOIN read_parquet('{cand}') c ON t.source1_entity_id=c.source1_entity_id AND t.matched_entity_id=c.candidate_entity_id").fetchone()[0]
    by_source=con.execute(f"SELECT CASE WHEN t.matched_entity_id LIKE 'S2-%' THEN 'S2' ELSE 'S3' END src,count(*) n FROM truth_cov_df t JOIN read_parquet('{cand}') c ON t.source1_entity_id=c.source1_entity_id AND t.matched_entity_id=c.candidate_entity_id GROUP BY 1").fetchall()
    BLOCKING_COVERAGE[split]={'truth_pairs':total,'captured_positive_pairs':captured,'blocked_out_positive_pairs':total-captured,'pair_coverage':captured/total if total else None,'captured_by_target_source':dict(by_source)}
    con.unregister('truth_cov_df')
print(json.dumps(BLOCKING_COVERAGE,indent=2))
(RUN_ROOT/'diagnostics/blocking_coverage.json').write_text(json.dumps(BLOCKING_COVERAGE,indent=2))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{
  "train": {
    "truth_pairs": 6489001,
    "captured_positive_pairs": 1855032,
    "blocked_out_positive_pairs": 4633969,
    "pair_coverage": 0.28587328003185697,
    "captured_by_target_source": {
      "S3": 881907,
      "S2": 973125
    }
  },
  "validation": {
    "truth_pairs": 1149364,
    "captured_positive_pairs": 329001,
    "blocked_out_positive_pairs": 820363,
    "pair_coverage": 0.28624613264379256,
    "captured_by_target_source": {
      "S3": 156474,
      "S2": 172527
    }
  }
}


507

In [18]:
# Fit record/token frequencies from train-split S1 plus the train target corpus only.
# Other splits transform with these frozen maps; unknown values map to zero/unseen flags.
TRAIN_STATS={}
# Base feature registry (all are pairwise evidence, not match decisions).
FEATURE_REGISTRY={}
def register(name,family,description,inputs,dtype='float32',fit_scope='stateless',cost='low',optional=False):
    FEATURE_REGISTRY[name]={'family':family,'description':description,'input_fields':inputs,'data_type':dtype,'fit_scope':fit_scope,'learned_from_train_only':fit_scope!='stateless','uses_external_data':False,'computation_cost':cost,'optional':optional}
for col in ['name','address']:
    for feat,desc in [('raw_exact','raw string equality'),('norm_exact','normalized string equality'),('sorted_exact','sorted token string equality'),('levenshtein','normalized Levenshtein similarity'),('damerau_levenshtein','normalized Damerau-Levenshtein similarity'),('jaro','Jaro similarity'),('jaro_winkler','Jaro-Winkler similarity'),('rapidfuzz_ratio','RapidFuzz ratio'),('rapidfuzz_wratio','RapidFuzz WRatio'),('token_jaccard','token set Jaccard'),('token_dice','token set Dice'),('overlap_coefficient','token overlap coefficient'),('containment_s1','S1 token containment'),('containment_candidate','candidate token containment'),('intersection_count','shared token count'),('token_alignment_mean','mean best token similarity'),('token_alignment_max','maximum best token similarity'),('length_ratio','shorter/longer raw length')]: register(f'{col}_{feat}',col,desc,[col+'_s1',col+'_candidate'],cost='medium' if 'alignment' in feat else 'low')
for feat,desc in [('country_equal','country agreement'),('country_both_present','both countries present'),('candidate_is_s2','candidate comes from S2'),('candidate_is_s3','candidate comes from S3'),('name_missing_either','name missing on either side'),('address_missing_either','address missing on either side'),('numeric_jaccard','address numeric-token Jaccard'),('numeric_intersection','shared address numeric token count'),('name_address_interaction','name and address both exactly agree'),('blocking_rule_count','number of blocking rules retrieving pair')]: register(feat,'context',desc,['pair fields'])
(RUN_ROOT/'metadata/feature_registry.json').write_text(json.dumps(FEATURE_REGISTRY,indent=2))
print('Registered core features:',len(FEATURE_REGISTRY))

Registered core features: 46


In [19]:
from rapidfuzz import fuzz
from rapidfuzz.distance import Levenshtein, DamerauLevenshtein, Jaro, JaroWinkler

def safe_text(x):
    if x is None or pd.isna(x): return ''
    return str(x)
def tokenize(value):
    s=safe_text(value)
    return [t for t in re.findall(r'\w+',s,flags=re.UNICODE) if t]
def numbers(value):
    return set(re.findall(r'\d+',safe_text(value)))
def normalized_edit_similarity(a,b):
    if not a or not b: return np.nan
    return float(Levenshtein.normalized_similarity(a,b))
def pair_features(row):
    f={}
    for key in ['name','address']:
        a=safe_text(row.get(key+'_raw_s1')); b=safe_text(row.get(key+'_raw_candidate'))
        an=safe_text(row.get(key+'_norm_s1')) or unicodedata.normalize('NFKC',a).casefold().strip()
        bn=safe_text(row.get(key+'_norm_candidate')) or unicodedata.normalize('NFKC',b).casefold().strip()
        at=set(tokenize(row.get(key+'_tokens_s1')) or tokenize(an)); bt=set(tokenize(row.get(key+'_tokens_candidate')) or tokenize(bn))
        inter=at&bt; union=at|bt
        f[key+'_raw_exact']=int(bool(a and b and a==b))
        f[key+'_norm_exact']=int(bool(an and bn and an==bn))
        sa=safe_text(row.get(key+'_sorted_s1')); sb=safe_text(row.get(key+'_sorted_candidate'))
        f[key+'_sorted_exact']=int(bool(sa and sb and sa==sb)) if sa or sb else np.nan
        f[key+'_levenshtein']=normalized_edit_similarity(an,bn)
        f[key+'_damerau_levenshtein']=float(DamerauLevenshtein.normalized_similarity(an,bn)) if an and bn else np.nan
        f[key+'_jaro']=float(Jaro.normalized_similarity(an,bn)) if an and bn else np.nan
        f[key+'_jaro_winkler']=float(JaroWinkler.normalized_similarity(an,bn)) if an and bn else np.nan
        f[key+'_rapidfuzz_ratio']=float(fuzz.ratio(an,bn)/100.0) if an and bn else np.nan
        f[key+'_rapidfuzz_wratio']=float(fuzz.WRatio(an,bn)/100.0) if an and bn else np.nan
        f[key+'_token_jaccard']=len(inter)/len(union) if union else np.nan
        f[key+'_token_dice']=2*len(inter)/(len(at)+len(bt)) if at or bt else np.nan
        f[key+'_overlap_coefficient']=len(inter)/min(len(at),len(bt)) if at and bt else np.nan
        f[key+'_containment_s1']=len(inter)/len(at) if at else np.nan
        f[key+'_containment_candidate']=len(inter)/len(bt) if bt else np.nan
        f[key+'_intersection_count']=len(inter)
        f[key+'_length_ratio']=min(len(an),len(bn))/max(len(an),len(bn)) if an and bn else np.nan
        # Bounded token alignment: candidate pairs only, strings capped to control pathological records.
        if key=='name' and at and bt and len(at)<=30 and len(bt)<=30:
            smaller,larger=(list(at),list(bt)) if len(at)<=len(bt) else (list(bt),list(at))
            best=[max(fuzz.ratio(x,y)/100 for y in larger) for x in smaller]
            f[key+'_token_alignment_mean']=float(np.mean(best)); f[key+'_token_alignment_max']=float(np.max(best))
        else: f[key+'_token_alignment_mean']=np.nan; f[key+'_token_alignment_max']=np.nan
    ca=safe_text(row.get('country_norm_s1')) or safe_text(row.get('country_s1')).casefold().strip()
    cb=safe_text(row.get('country_norm_candidate')) or safe_text(row.get('country_candidate')).casefold().strip()
    f['country_equal']=int(bool(ca and cb and ca==cb)); f['country_both_present']=int(bool(ca and cb))
    f['candidate_is_s2']=int(row.get('candidate_source')=='S2'); f['candidate_is_s3']=int(row.get('candidate_source')=='S3')
    na=safe_text(row.get('name_raw_s1')); nb=safe_text(row.get('name_raw_candidate'))
    aa=safe_text(row.get('address_raw_s1')); ab=safe_text(row.get('address_raw_candidate'))
    f['name_missing_either']=int(not na or not nb); f['address_missing_either']=int(not aa or not ab)
    nums_a=numbers(row.get('address_numbers_s1')) or numbers(aa); nums_b=numbers(row.get('address_numbers_candidate')) or numbers(ab)
    f['numeric_jaccard']=len(nums_a&nums_b)/len(nums_a|nums_b) if nums_a|nums_b else np.nan
    f['numeric_intersection']=len(nums_a&nums_b)
    f['name_address_interaction']=int(f['name_norm_exact'] and f['address_norm_exact'])
    if globals().get('VECTORIZERS'):
        from sklearn.preprocessing import normalize
        for field,prefix in [('name','name_norm'),('address','address_norm')]:
            left=safe_text(row.get(prefix+'_s1')); right=safe_text(row.get(prefix+'_candidate'))
            for analyzer in ['word','char','char_wb']:
                key=f'{prefix}_{analyzer}'; vec=VECTORIZERS.get(key)
                if vec is None:
                    f[f'{field}_tfidf_{analyzer}_cosine']=np.nan
                elif left and right:
                    mats=vec.transform([left,right])
                    f[f'{field}_tfidf_{analyzer}_cosine']=float(mats[0].multiply(mats[1]).sum())
                else:
                    f[f'{field}_tfidf_{analyzer}_cosine']=np.nan
    prov=safe_text(row.get('rules_triggered')); f['blocking_rule_count']=len([x for x in prov.split(',') if x.strip()]) if prov else np.nan
    # Unicode/script/corruption indicators are diagnostic, never corrective.
    combined=na+' '+nb+' '+aa+' '+ab
    f['non_ascii_char_fraction']=sum(ord(ch)>127 for ch in combined)/len(combined) if combined else 0.0
    f['replacement_char_count']=combined.count('�')
    f['mojibake_marker_count']=sum(combined.count(x) for x in ['Ã','Â','â€','ï¿½'])
    return f


In [20]:
# Build narrow lookup relations, then stream candidate rows with both records attached.
# Missing optional schema fields project as NULL; no unrelated-field substitution.
def projection(table,side):
    m=RECORD_MAP[table]; out=[]
    for dest,src in [('id','id'),('name_raw','name_raw'),('name_norm','name_norm'),('name_tokens','name_tokens'),('name_sorted','name_sorted'),('name_numbers','name_numbers'),('address_raw','address_raw'),('address_norm','address_norm'),('address_tokens','address_tokens'),('address_sorted','address_sorted'),('address_numbers','address_numbers'),('country','country'),('country_norm','country_norm')]:
        col=m[src]
        expr=f'CAST({chr(34)+col+chr(34)} AS VARCHAR)' if col else 'NULL::VARCHAR'
        out.append(f'{expr} AS {dest}_{side}')
    return ','.join(out)

def ensure_views():
    for t in RECORD_PATHS:
        con.execute(f"CREATE OR REPLACE VIEW {t} AS SELECT {projection(t,'')} FROM read_parquet('{RECORD_PATHS[t]}')")
# Views with standardized alias fields.
for t,p in RECORD_PATHS.items():
    m=RECORD_MAP[t]; select=[]
    for logical in ['id','name_raw','name_norm','name_tokens','name_sorted','name_numbers','address_raw','address_norm','address_tokens','address_sorted','address_numbers','country','country_norm']:
        col=m[logical]
        if col: select.append(f'CAST("{col}" AS VARCHAR) AS {logical}')
        else: select.append(f'NULL::VARCHAR AS {logical}')
    con.execute(f"CREATE OR REPLACE VIEW {t} AS SELECT {','.join(select)} FROM read_parquet('{p}')")

def candidate_query(split):
    p=CANONICAL[split]
    targets="""SELECT * FROM train_source2 UNION ALL SELECT * FROM train_source3""" if split!='test' else """SELECT * FROM test_source2 UNION ALL SELECT * FROM test_source3"""
    # explicit fields from view records
    target_views=['train_source2','train_source3'] if split!='test' else ['test_source2','test_source3']
    targetsql=' UNION ALL '.join([f"SELECT *, CASE WHEN id LIKE 'S2-%' THEN 'S2' ELSE 'S3' END AS candidate_source FROM {t}" for t in target_views])
    s1view='train_source1' if split!='test' else 'test_source1'
    return f"SELECT c.source1_entity_id,c.candidate_entity_id,c.candidate_source,c.rules_triggered, {', '.join('a.'+x+' AS '+x+'_s1' for x in ['name_raw','name_norm','name_tokens','name_sorted','name_numbers','address_raw','address_norm','address_tokens','address_sorted','address_numbers','country','country_norm'])}, {', '.join('b.'+x+' AS '+x+'_candidate' for x in ['name_raw','name_norm','name_tokens','name_sorted','name_numbers','address_raw','address_norm','address_tokens','address_sorted','address_numbers','country','country_norm'])} FROM read_parquet('{p}') c JOIN {s1view} a ON c.source1_entity_id=a.id JOIN ({targetsql}) b ON c.candidate_entity_id=b.id"

# Label map is added only for train/validation. Test is never joined to labels.
positive_keys=set(zip(positives.source1_entity_id.astype(str),positives.matched_entity_id.astype(str)))
def checkpoint_path(split,shard): return RUN_ROOT/'checkpoints'/f'{split}_shard_{shard:06d}.json'
def atomic_parquet(df,path):
    tmp=path.with_suffix(path.suffix+'.tmp')
    df.to_parquet(tmp,index=False)
    pq.ParquetFile(tmp)  # validate readable/schema before replace
    tmp.replace(path)

def feature_split(split):
    dest=RUN_ROOT/split; dest.mkdir(parents=True,exist_ok=True)
    q=candidate_query(split)
    cur=con.execute(q); total=0; shard=0; start=time.time()
    while True:
        batch=cur.fetch_df_chunk(SHARD_ROWS)
        if batch is None or batch.empty: break
        ck=checkpoint_path(split,shard); out=dest/f'pair_features_{split}-{shard:05d}.parquet'
        if RESUME and ck.exists() and out.exists():
            meta=json.loads(ck.read_text())
            if meta.get('status')=='complete' and meta.get('input_path')==str(CANONICAL[split]) and meta.get('rows')==len(batch):
                total+=len(batch); shard+=1; continue
        rows=[]
        for row in batch.to_dict('records'):
            feats=pair_features(row)
            outrow={k:row[k] for k in ['source1_entity_id','candidate_entity_id','candidate_source','rules_triggered']}
            outrow.update(feats)
            if split!='test':
                outrow['label']=int((str(row['source1_entity_id']),str(row['candidate_entity_id'])) in positive_keys)
            rows.append(outrow)
        feature_df=pd.DataFrame(rows)
        atomic_parquet(feature_df,out)
        meta={'split':split,'shard':shard,'input_path':str(CANONICAL[split]),'output_path':str(out),'rows':len(feature_df),'feature_columns':[c for c in feature_df if c not in ['source1_entity_id','candidate_entity_id','candidate_source','rules_triggered','label']],'runtime_seconds':time.time()-start,'status':'complete','random_seed':RANDOM_SEED}
        ck.write_text(json.dumps(meta,indent=2))
        total+=len(feature_df); shard+=1
        if shard%10==0: print('[CHECKPOINT]',split,'rows',total,'pairs/sec',round(total/max(time.time()-start,1),1))
    return {'split':split,'rows':total,'shards':shard,'runtime_seconds':time.time()-start}


In [21]:
if ENABLE_TFIDF:
    from sklearn.feature_extraction.text import TfidfVectorizer
    import joblib
    TFIDF_MAX_DOCS=200_000  # measured/capped training corpus sample, never validation/test
    TFIDF_MAX_FEATURES=100_000
    train_s1_path=RECORD_PATHS['train_source1']; train_s2_path=RECORD_PATHS['train_source2']; train_s3_path=RECORD_PATHS['train_source3']
    con.register('train_ids_df',pd.DataFrame({'id':list(train_s1)}))
    docs={}
    for field in ['name_norm','address_norm']:
        pieces=[]
        per_source=max(1,TFIDF_MAX_DOCS//3)
        # Deterministic, train-only bounded corpus; unsupervised validation/test text is excluded from fit.
        for t in ['train_source1','train_source2','train_source3']:
            query=f"SELECT CAST(r.{field} AS VARCHAR) txt FROM {t} r"
            if t=='train_source1': query += " JOIN train_ids_df i ON r.id=i.id"
            query += f" WHERE r.{field} IS NOT NULL AND trim(CAST(r.{field} AS VARCHAR))<>'' ORDER BY hash(CAST(r.id AS VARCHAR) || '-{RANDOM_SEED}') LIMIT {per_source}"
            pieces.extend(con.execute(query).df().txt.fillna('').astype(str).tolist())
        docs[field]=pieces
    VECTORIZERS={}
    for field,analyzer,ngram_range in [('name_norm','word',(1,2)),('name_norm','char',(2,5)),('name_norm','char_wb',(2,5)),('address_norm','word',(1,2)),('address_norm','char',(2,5)),('address_norm','char_wb',(2,5))]:
        key=f'{field}_{analyzer}'
        vec=TfidfVectorizer(analyzer=analyzer,ngram_range=ngram_range,max_features=TFIDF_MAX_FEATURES,dtype=np.float32,sublinear_tf=True)
        try: vec.fit(docs[field])
        except ValueError as e:
            print('[SKIP]',key,'empty vocabulary:',e); continue
        VECTORIZERS[key]=vec
        joblib.dump(vec,RUN_ROOT/f'fit_artifacts/{key}.joblib')
    con.unregister('train_ids_df')
    (RUN_ROOT/'fit_artifacts/train_fit_complete.json').write_text(json.dumps({'fit_scope':'training S1 IDs and training S2/S3 records only','max_docs':TFIDF_MAX_DOCS,'actual_docs':{k:len(v) for k,v in docs.items()},'vectorizers':list(VECTORIZERS)},indent=2))
    print('[OK] Train-only TF-IDF fit:',list(VECTORIZERS))
else:
    VECTORIZERS={}
    (RUN_ROOT/'fit_artifacts/train_fit_complete.json').write_text(json.dumps({'fit_scope':'core features are stateless; TF-IDF disabled','vectorizers':[]},indent=2))
    print('[SKIP] TF-IDF disabled by configuration.')


[SKIP] TF-IDF disabled by configuration.


In [22]:
# Smoke test one actual joined batch before full generation; requires at least one candidate.
smoke_sql=candidate_query('train')
smoke=con.execute(smoke_sql+' LIMIT 16').df()
if smoke.empty: raise RuntimeError('Train final candidate set is empty; verify blocking artifacts and candidate schema.')
smoke_features=pd.DataFrame([pair_features(r) for r in smoke.to_dict('records')])
if smoke_features.empty: raise RuntimeError('Smoke test returned no features')
print('[OK] smoke rows',len(smoke),'feature count',len(smoke_features.columns),'S2/S3',smoke.candidate_source.value_counts().to_dict())
print('[WARN] Check smoke sample for missing/multilingual/positive cases; the notebook does not invent examples if absent.')
(RUN_ROOT/'diagnostics/smoke_test.json').write_text(json.dumps({'rows':len(smoke),'features':list(smoke_features.columns),'sources':smoke.candidate_source.value_counts().to_dict()},indent=2))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[OK] smoke rows 16 feature count 49 S2/S3 {'S2': 16}
[WARN] Check smoke sample for missing/multilingual/positive cases; the notebook does not invent examples if absent.


1451

In [23]:
# Generate full candidates in chunks. Set RUN_SPLITS explicitly after smoke test.
RUN_SPLITS=[]  # e.g. ['train','validation','test']; no full job starts automatically.
if set(RUN_SPLITS)-{'train','validation','test'}: raise ValueError(f'Invalid RUN_SPLITS: {RUN_SPLITS}')
if 'test' in RUN_SPLITS and not (RUN_ROOT/'fit_artifacts/train_fit_complete.json').exists():
    raise RuntimeError('Generate/freeze train-only fit artifacts and review validation before test feature generation.')
RUN_SUMMARY=[]
for split in RUN_SPLITS:
    result=feature_split(split); RUN_SUMMARY.append(result); print('[OK]',result)


In [24]:
if RUN_SUMMARY:
    all_paths=[]
    for split in RUN_SPLITS:
        all_paths.extend(sorted((RUN_ROOT/split).glob(f'pair_features_{split}-*.parquet')))
    quality=[]
    for p in all_paths:
        df=pd.read_parquet(p)
        feature_cols=[c for c in df.columns if c not in {'source1_entity_id','candidate_entity_id','candidate_source','rules_triggered','label'}]
        for c in feature_cols:
            x=pd.to_numeric(df[c],errors='coerce')
            quality.append({'file':str(p),'feature':c,'rows':len(df),'missing_fraction':float(x.isna().mean()),'infinite_count':int(np.isinf(x.dropna()).sum()) if len(x.dropna()) else 0,'unique_values_sample':int(x.nunique(dropna=True))})
    qdf=pd.DataFrame(quality); qdf.to_parquet(RUN_ROOT/'diagnostics/feature_quality_report.parquet',index=False)
    if (qdf.infinite_count>0).any(): raise RuntimeError('Infinite feature value found')
    print('Feature quality rows:',len(qdf))

# Candidate-loss audit and split-key leakage checks.
LEAKAGE={'train_validation_s1_overlap':bool(train_s1 & val_s1),'test_has_labels':False,'external_data_used':False,'candidate_pair_filtering':False,'learned_transform_scope':'none in core stateless features; optional TF-IDF disabled'}
if LEAKAGE['train_validation_s1_overlap']: raise RuntimeError('Source-1 train/validation leakage detected')
(RUN_ROOT/'diagnostics/leakage_audit.json').write_text(json.dumps(LEAKAGE,indent=2))
FAMILY_COVERAGE={}
for feat,meta in FEATURE_REGISTRY.items(): FAMILY_COVERAGE.setdefault(meta['family'],{'feature_names':[],'enabled':True,'fit_scope':meta['fit_scope']})['feature_names'].append(feat)
FAMILY_COVERAGE['tfidf']={'feature_names':[],'enabled':ENABLE_TFIDF,'reason':'disabled pending memory-budgeted train-only fit' if not ENABLE_TFIDF else 'enabled'}
(RUN_ROOT/'diagnostics/feature_family_coverage.json').write_text(json.dumps(FAMILY_COVERAGE,indent=2))
print(json.dumps(LEAKAGE,indent=2))

{
  "train_validation_s1_overlap": false,
  "test_has_labels": false,
  "external_data_used": false,
  "candidate_pair_filtering": false,
  "learned_transform_scope": "none in core stateless features; optional TF-IDF disabled"
}


In [25]:
# Per-split candidate counts must exactly match feature rows; no candidates may disappear.
FINAL_COUNTS={}
for split in ['train','validation','test']:
    source_count=con.execute(f"SELECT count(*) FROM read_parquet('{CANONICAL[split]}')").fetchone()[0]
    files=sorted((RUN_ROOT/split).glob(f'pair_features_{split}-*.parquet'))
    output_count=sum(pq.ParquetFile(p).metadata.num_rows for p in files)
    if split in RUN_SPLITS and source_count!=output_count:
        raise RuntimeError(f'{split} candidate loss: input {source_count}, feature rows {output_count}')
    FINAL_COUNTS[split]={'candidate_rows':source_count,'feature_rows':output_count,'shards':len(files),'preserved':source_count==output_count if split in RUN_SPLITS else None}
print(FINAL_COUNTS)
manifest={'run_version':RUN_VERSION,'timestamp_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),'source_artifacts':{**{k:str(v) for k,v in RECORD_PATHS.items()},**{k:str(v) for k,v in CANDIDATE_PATHS.items()},'split':str(SPLIT_PATH),'positives':[str(p) for p in POSITIVE_PATHS]},'schemas':SCHEMA_INVENTORY,'row_counts':FINAL_COUNTS,'feature_count':len(FEATURE_REGISTRY),'feature_families':sorted({x['family'] for x in FEATURE_REGISTRY.values()}),'fit_scopes':{'core':'stateless','tfidf':'disabled'},'random_seed':RANDOM_SEED,'software':RESOURCE_PROFILE['packages'],'optional_modules':{'tfidf':ENABLE_TFIDF,'phonetic':False,'embeddings':False},'resource_observations':RESOURCE_PROFILE,'warnings':['TF-IDF disabled pending memory-budgeted train-only fit.','Phonetic and semantic embeddings not implemented in this baseline.','Run candidate output path selection explicitly after reviewing inventory.']}
(RUN_ROOT/'metadata/run_manifest.json').write_text(json.dumps(manifest,indent=2,default=str))
(RUN_ROOT/'manifests/run_manifest.json').write_text(json.dumps(manifest,indent=2,default=str))
if WRITE_SUCCESS and set(RUN_SPLITS)=={'train','validation','test'}:
    success={'status':'SUCCESS','timestamp_utc':manifest['timestamp_utc'],'notebook':'Amazon_ML_Challenge_2026_Feature_Engineering.ipynb','run_version':RUN_VERSION,'counts':FINAL_COUNTS,'warning':'SUCCESS marks completed configured splits only; optional modules may be disabled.'}
    (RUN_ROOT/'SUCCESS.json').write_text(json.dumps(success,indent=2))
    print('STEP 3 FEATURE ENGINEERING COMPLETE')
    print('Train:',RUN_ROOT/'train','Validation:',RUN_ROOT/'validation','Test:',RUN_ROOT/'test','Fit artifacts:',RUN_ROOT/'fit_artifacts','Registry:',RUN_ROOT/'metadata/feature_registry.json','Diagnostics:',RUN_ROOT/'diagnostics','Manifest:',RUN_ROOT/'metadata/run_manifest.json')
else:
    print('[WARN] All three splits (train, validation, test) must complete before SUCCESS.json is written.')

{'train': {'candidate_rows': 18903790, 'feature_rows': 0, 'shards': 0, 'preserved': None}, 'validation': {'candidate_rows': 3348934, 'feature_rows': 0, 'shards': 0, 'preserved': None}, 'test': {'candidate_rows': 11408860, 'feature_rows': 0, 'shards': 0, 'preserved': None}}
[WARN] All three splits (train, validation, test) must complete before SUCCESS.json is written.
